## ganda-grammar, carried by this notebook

This copy of `build_dictionary.ipynb` carries ganda-grammar (commit 90db4d1-dirty). The cells below write its package (`src/`, as
`ganda_grammar`) and `scripts/` to `/tmp/ganda_grammar_bundle` and put that first on `sys.path`, so the notebook needs no copy of the
repository as an input. Made by `scripts/bundle_notebook.py`; rebuild it after changing the code.


In [1]:
import sys
from pathlib import Path

BUNDLE = Path("/tmp/ganda_grammar_bundle")
for folder in ['/tmp/ganda_grammar_bundle/ganda_grammar', '/tmp/ganda_grammar_bundle/ganda_grammar/model', '/tmp/ganda_grammar_bundle/ganda_grammar/rules', '/tmp/ganda_grammar_bundle/scripts']:
    Path(folder).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BUNDLE))


In [2]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/__init__.py
"""ganda-grammar: Luganda spelling and grammar correction, by cited rules or by a fine-tuned ByT5-small model."""
from .rules import RULES, Edit, Result
from .rules import correct as _correct_rules

__version__ = "0.1.0"


def correct(text: str, engine: str = "rules", **model_options) -> Result:
    """Correct Luganda text. engine="rules" (default), or "model" after pip install 'ganda-grammar[model]'.

    model_options go to the model engine: model (Hugging Face id or path), device, batch_size, num_beams.
    """
    if engine == "rules":
        return _correct_rules(text)
    if engine == "model":
        from .model import correct as _correct_model
        return _correct_model(text, **model_options)
    raise ValueError(f"engine must be 'rules' or 'model', not {engine!r}")


__all__ = ["RULES", "Edit", "Result", "correct"]


Writing /tmp/ganda_grammar_bundle/ganda_grammar/__init__.py


In [3]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/cli.py
"""ganda-grammar correct: correct Luganda text. ganda-grammar mine: collect corrections from a corpus."""
import argparse
import json
import sys
from dataclasses import asdict
from itertools import chain


def main(argv=None):
    ap = argparse.ArgumentParser(prog="ganda-grammar", description=__doc__)
    sub = ap.add_subparsers(dest="cmd", required=True)

    c = sub.add_parser("correct", help="correct text given as an argument, or on stdin")
    c.add_argument("text", nargs="?")
    c.add_argument("--engine", choices=("rules", "model"), default="rules")
    c.add_argument("--model", help="Hugging Face id or local path of the model (--engine model)")
    c.add_argument("--json", action="store_true", help="print the corrected text and its edits as JSON")

    m = sub.add_parser("mine", help="run the rules over a corpus, write one JSONL row per corrected sentence")
    m.add_argument("paths", nargs="*", help=".txt and .jsonl files, or folders of them")
    m.add_argument("--hf", help="also stream this Hugging Face dataset, e.g. henryhale/luganda-pretrain2")
    m.add_argument("--split", default="train")
    m.add_argument("--exclude-source", action="append", default=[], metavar="SOURCE",
                   help="skip dataset rows whose source column is SOURCE (repeatable)")
    m.add_argument("--field", default="text", help="text field of .jsonl rows and dataset rows (default: text)")
    m.add_argument("--limit", type=int, help="stop after this many corrected sentences")
    m.add_argument("-o", "--out", default="corrections/corrections.jsonl",
                   help="output file; keep it in corrections/, the folder that is published (default: %(default)s)")

    a = ap.parse_args(argv)
    if a.cmd == "correct":
        from . import correct

        text = sys.stdin.read() if a.text is None else a.text
        r = correct(text, a.engine, **({"model": a.model} if a.model else {}))
        if a.json:
            print(json.dumps({"text": r.text, "edits": [asdict(e) for e in r.edits]}, ensure_ascii=False, indent=2))
        else:
            print(r.text, end="" if a.text is None else "\n")
    else:
        from .mine import hf_texts, local_texts, mine

        if not a.paths and not a.hf:
            ap.error("mine needs paths, --hf, or both")
        texts = chain(local_texts(a.paths, a.field),
                      hf_texts(a.hf, a.split, a.field, set(a.exclude_source)) if a.hf else ())
        for rule, n in mine(texts, a.out, a.limit).most_common():
            print(f"{rule}\t{n}")
        print(f"wrote {a.out}", file=sys.stderr)


if __name__ == "__main__":
    main()


Writing /tmp/ganda_grammar_bundle/ganda_grammar/cli.py


In [4]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/mine.py
"""Mine corrections: run the rules over a corpus and write one JSONL row per corrected sentence.

Row: {"source", "original", "corrected", "edits": [{"rule", "start", "end", "before", "after"}]}, with offsets
into "original". Repeated sentences are written once.
"""
import json
from collections import Counter
from dataclasses import asdict
from pathlib import Path

from .rules import correct
from .rules.base import sentences


def local_texts(paths, field="text"):
    """(source, text) from .txt files (whole file) and .jsonl files (each row's field); folders are searched."""
    for p in map(Path, paths):
        for f in sorted(x for x in p.rglob("*") if x.suffix in (".txt", ".jsonl")) if p.is_dir() else [p]:
            if f.suffix == ".jsonl":
                for line in f.open(encoding="utf-8"):
                    row = json.loads(line) if line.strip() else {}
                    if isinstance(row, dict) and isinstance(row.get(field), str):
                        yield str(f), row[field]
            else:
                yield str(f), f.read_text(encoding="utf-8", errors="replace")


def hf_texts(repo, split="train", field="text", exclude_sources=()):
    """(source, text) streamed from a Hugging Face dataset, skipping rows whose "source" is in exclude_sources."""
    try:
        from datasets import load_dataset
    except ImportError as e:
        raise ImportError("mining a Hugging Face dataset needs: pip install datasets") from e
    for row in load_dataset(repo, split=split, streaming=True):
        src = row.get("source")
        if src not in exclude_sources:
            yield f"{repo}:{src}" if src else repo, row[field]


def mine(texts, out, limit=None):
    """Write the corrected sentences of (source, text) pairs to out; return the edit count per rule."""
    # ponytail: one process; split the input across processes if a Kaggle session runs out of time
    seen, counts, rows = set(), Counter(), 0
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as f:
        for source, text in texts:
            for s, e in sentences(text):
                sentence = text[s:e]
                if hash(sentence) in seen:
                    continue
                r = correct(sentence)
                if not r.edits:
                    continue
                seen.add(hash(sentence))
                row = {"source": source, "original": sentence, "corrected": r.text, "edits": [asdict(x) for x in r.edits]}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                counts.update(x.rule for x in r.edits)
                rows += 1
                if rows == limit:
                    return counts
    return counts


Writing /tmp/ganda_grammar_bundle/ganda_grammar/mine.py


In [5]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/model/__init__.py
"""Model engine: google/byt5-small fine-tuned to correct Luganda sentences (see notebooks/train_byt5.ipynb).

Each sentence is rewritten by the model; the edits are the character differences, with rule "MODEL".
"""
import difflib
from functools import lru_cache

from ..rules.base import Edit, Result, apply, sentences

MODEL = "henryhale/ganda-grammar-byt5-small"
# ponytail: a rewrite further from the input than this is taken as a hallucination and dropped; tune on a dev set
MIN_SIMILARITY = 0.8


@lru_cache(maxsize=None)
def load(model=MODEL, device=None):
    try:
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    except ImportError as e:
        raise ImportError("the model engine needs: pip install 'ganda-grammar[model]'") from e
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    return AutoTokenizer.from_pretrained(model), AutoModelForSeq2SeqLM.from_pretrained(model).to(device).eval()


def correct(text, model=MODEL, device=None, batch_size=16, num_beams=4):
    import torch

    tok, net = load(model, device)
    spans = list(sentences(text))
    outputs = []
    for i in range(0, len(spans), batch_size):
        batch = tok([text[s:e] for s, e in spans[i:i + batch_size]], return_tensors="pt", padding=True).to(net.device)
        with torch.no_grad():  # ByT5 works on bytes: the output is about as long as the input
            ids = net.generate(**batch, num_beams=num_beams, max_new_tokens=int(batch.input_ids.shape[1] * 1.2) + 8)
        outputs += tok.batch_decode(ids, skip_special_tokens=True)
    edits = []
    for (s, e), new in zip(spans, outputs):
        old = text[s:e]
        diff = difflib.SequenceMatcher(None, old, new, autojunk=False)
        if new != old and diff.ratio() >= MIN_SIMILARITY:
            edits += [Edit("MODEL", s + i1, s + i2, old[i1:i2], new[j1:j2])
                      for op, i1, i2, j1, j2 in diff.get_opcodes() if op != "equal"]
    return Result(apply(text, edits), edits)


Writing /tmp/ganda_grammar_bundle/ganda_grammar/model/__init__.py


In [6]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/__init__.py
"""Rule engine. Each family module has RULES (id → description, source) and edits(text, tokens) → [Edit]."""
from . import agreement, elision, joining, orthography
from .base import Edit, Result, apply, luganda_tokens, sentences

FAMILIES = (agreement, elision, joining, orthography)  # where two edits overlap, the earlier family's wins
RULES = {rid: rule for f in FAMILIES for rid, rule in f.RULES.items()}


def correct(text):
    found = []
    for start, end in sentences(text):
        toks = luganda_tokens(text, start, end)
        for f in FAMILIES:
            found += f.edits(text, toks)
    kept = []
    for e in found:
        if all(e.end <= k.start or e.start >= k.end for k in kept):
            kept.append(e)
    kept.sort(key=lambda e: e.start)
    return Result(apply(text, kept), kept)


__all__ = ["RULES", "Edit", "Result", "correct"]


Writing /tmp/ganda_grammar_bundle/ganda_grammar/rules/__init__.py


In [7]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/agreement.py
"""Agreement: an adjective or possessive right after a noun takes the noun's class prefix.

Only after the nouns listed here, whose class is certain. A prefix alone isn't enough: eki-/obu-/ama- also start
relative verbs (ekisinga obunene) and ordinary verbs (amanyi = he knows).
"""
from .base import APOSTROPHES, Edit
from .elision import TRIGGERS

SRC = "Sternefeld & Sseguya, A Luganda Grammar (2015)"
RULES = {
    "AGREE_ADJ": ("an adjective takes its noun's class: ebintu ekirungi → ebintu ebirungi", f"{SRC}, §7"),
    "AGREE_POSS": ("a possessive takes its noun's class: ebitabo kyange → ebitabo byange", f"{SRC}, §6"),
}


def _pairs(sg, pl, stems):
    """Nouns of a singular/plural class pair: eki + tabo = ekitabo, eky + alo = ekyalo before a vowel."""
    glide = {"i": "y", "u": "w"}
    return {(pre[:-1] + glide[pre[-1]] if s[0] in "aeiou" else pre) + s: cls
            for s in stems.split() for cls, pre in (sg, pl)}


NOUNS = {
    **_pairs(("7", "eki"), ("8", "ebi"), """ntu fo biina seera tongole tundu gambo buga zibu ragiro rwadde faananyi
        binja gendererwa zimbe rabo wandiiko tabo kopo saawe bonerezo teeso rowoozo senge tone kwekweto rooto buuzo
        banja sanja rungo gere tanda muli bira go nnya kolwa ka alo uma ambalo"""),
    **_pairs(("12", "aka"), ("14", "obu"), """kiiko lulu wuka tale bonero benje tambi tabo baga kodyo faananyi piira
        sente waayiro ti solya yumba ntu papula zannyo goye veera wunga gaali wale sawo tebe buga sozi saale cupa
        pande motoka duuka kebe seera"""),
    **dict.fromkeys("""obulamu obugagga obwakabaka obukulembeze obujulizi obubaka obukuumi obusobozi obwenkanya
        obulwadde obuvunaanyizibwa""".split(), "14"),
    **dict.fromkeys("""olunaku olugendo olusozi olulimi oluggi olubuto olupapula olugoye olutalo oluyimba oluguudo
        olukiiko olusuku olususu olukalala olugero olubiri oluzzi olunyiriri olwazi""".split(), "11"),
    # Not loanwords with e + double consonant (essanduuko, essimu, ebbaluwa, ekkanisa, eggaali): they are class 9
    **dict.fromkeys("""erinnya eriiso ejjinja essomero eddoboozi ettooke eggwanga ekkubo eggye eggi etteeka essaza
        eddagala ettendekero ekkomera essomo eddwaliro""".split(), "5"),
    **dict.fromkeys("""amannya amaaso amayinja amasomero amaloboozi amatooke amawanga amakubo amagye amagi amateeka
        amasaza amalagala amatendekero amakomera amasomo amalwaliro amazzi amata amafuta amaanyi amagezi amawulire
        amaka amaziga amazima""".split(), "6"),
}

# Adjective prefixes with the augment (§7). Class 5 is left out: its forms are irregular (eddene, eddungi, eribi).
ADJ_PREFIX = {"7": "eki", "8": "ebi", "12": "aka", "14": "obu", "11": "olu", "6": "ama"}
ADJ_STEMS = "nene lungi bi kulu tono to nafu siru lalu lamu kadde gezi mpi gazi zito ddugavu myufu kalu bisi tuufu ngi pya"


def adjective(cls, stem):
    pre = ADJ_PREFIX[cls]
    return pre + ("r" + stem[1:] if pre.endswith("i") and stem[0] == "l" else stem)  # ki + lungi = kirungi


# Forms that can be wrong after a noun of another class. Not obu- forms, which are also adverbs (akola obulungi),
# nor forms that are also nouns: akatono 'a little', akabi 'danger', akakadde 'million', amakulu 'meaning',
# amato 'boats', amagezi 'wisdom'.
WRONG_ADJ = {adjective(c, s): (c, s) for c in ADJ_PREFIX for s in ADJ_STEMS.split() if c != "14"}
for w in "akatono akabi akakadde amakulu amato amagezi".split():
    del WRONG_ADJ[w]

# Possessive prefixes before the stem's a (§6). Classes 1 and 2 are left out: wange, bange are also vocatives.
POSS_PREFIX = {"3": "gw", "4": "gy", "5": "ly", "6": "g", "7": "ky", "8": "by", "9": "y", "10": "z", "11": "lw",
               "12": "k", "13": "tw", "14": "bw", "15": "kw"}


def possessive(cls, stem):
    pre = POSS_PREFIX[cls]
    return pre + (stem[1:] if stem == "aabwe" and pre[-1] in "wy" else stem)  # kyabwe but gaabwe (§1.1.6)


POSS = {POSS_PREFIX[c] + s: (c, "aabwe" if s == "abwe" else s)
        for c in POSS_PREFIX for s in ("ange", "affe", "ammwe", "aabwe", "abwe")}


def edits(text, toks):
    out = []
    for prev, a, b in zip([None] + toks, toks, toks[1:]):
        cls = NOUNS.get(a.text[0].lower() + a.text[1:])
        # Skip y'ekibuga ebirungi and bya ekibiina ebirungi: the adjective may belong to an earlier noun.
        if (not cls or a.end == b.start or text[a.end:b.start].strip(" \t")
                or (a.start and text[a.start - 1] in APOSTROPHES + "-")
                or (prev and prev.text.lower() in TRIGGERS)):
            continue
        if (hit := WRONG_ADJ.get(b.text)) and hit[0] != cls and cls in ADJ_PREFIX:
            out.append(Edit("AGREE_ADJ", b.start, b.end, b.text, adjective(cls, hit[1])))
        elif (hit := POSS.get(b.text)) and hit[0] != cls:
            out.append(Edit("AGREE_POSS", b.start, b.end, b.text, possessive(cls, hit[1])))
    return out


Writing /tmp/ganda_grammar_bundle/ganda_grammar/rules/agreement.py


In [8]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/base.py
"""What every rule family shares: edits, sentences, tokens, and the check that a word or sentence is Luganda."""
import re
from dataclasses import dataclass, field
from typing import NamedTuple

APOSTROPHES = "'’ʼ‘"


@dataclass(frozen=True)
class Edit:
    rule: str  # rule id, see RULES; "MODEL" for the model engine
    start: int  # character offsets into the original text
    end: int
    before: str
    after: str


@dataclass
class Result:
    text: str
    edits: list = field(default_factory=list)


class Token(NamedTuple):
    start: int
    end: int
    text: str


def apply(text, edits):
    """Apply non-overlapping edits, sorted by start."""
    out, pos = [], 0
    for e in edits:
        out += [text[pos:e.start], e.after]
        pos = e.end
    return "".join(out) + text[pos:]


SENTENCE = re.compile(r"[^\s.!?][^.!?\n]*[.!?]*")
WORD = re.compile(r"[^\W\d_]+")


def sentences(text):
    """(start, end) of each sentence, without surrounding whitespace."""
    for m in SENTENCE.finditer(text):
        yield m.start(), m.start() + len(m[0].rstrip())


# A Luganda word is a run of syllables (consonant cluster + short or long vowel) and ends in a vowel. Only the first
# syllable may start with a vowel: two different vowels in a row (believe, milieu) aren't written in Luganda.
GEMINATE = "(?:bb|cc|dd|ff|gg|jj|kk|mm|nn|pp|ss|tt|vv|zz|ŋŋ)"
NC = "(?:m[bpfv]|n[dtkzfgjsvc])"  # nasal + consonant (Walusimbi 2015, §1.1.5, §1.1.7)
ONSET = f"(?:(?:nny|ny|{GEMINATE}|{NC}|[bcdfgjklmnprstvzŋ])[wy]?|[wy])"
VOWEL = "(?:aa|ee|ii|oo|uu|[aeiou])"
SHAPE = re.compile(f"{VOWEL}?(?:{ONSET}{VOWEL})*")

# English words with a Luganda shape that a spelling rule would change, found in henryhale/luganda-pretrain2
ENGLISH = frozenset("""
mobile believe file milieu twee failure release tile mile missile silence angelo vile delaware relative envelope
bela relate elite ila accelerate delete jubilee milo bile simile deliverance nile eliminate timeline compile militia
relieve pile consuelo lilongwe delegate reliance elaborate facilitate pele voile leila utilize tensile silica cilia
dilute vila diligence affiliate mobilize deliberate relevance pipeline delicate fossilize silicate facile revile
sweetie silo reconcile futile dilemma vigilante elusive gentile defile resilience gila felice guile relocate
vigilance automobile delude antelope baseline docile vuvuzela volatile virile filipino smile while
""".split())

# Function words of languages that share Luganda's word shape (Swahili, Chichewa, Portuguese/Spanish) and of English,
# leaving out the ones that are also Luganda words (a, be, so, we)
FOREIGN = frozenset("""
ni katika kwamba lakini hii sana pia kuwa ambao ambaye hiyo wewe mimi yeye huo hilo wao sisi nini
anthu kuti ndi ndipo ine iye que ele ela dele pela pelo para uma com los las del por
the and of is are was were to in on at for with from by this that these those it its as an or not no have has had
will would can could which who what when where there here their they them he his she her you your our also only but if
than then into about after before
""".split())


def looks_luganda(word):
    """Lowercase, made of Luganda syllables, not a known English word, and not 30+ letters of glued-together text."""
    return word.islower() and len(word) <= 30 and word not in ENGLISH and SHAPE.fullmatch(word) is not None


def luganda_tokens(text, start, end):
    """The words of text[start:end], or [] when the sentence doesn't read as Luganda, so no rule touches it."""
    # ponytail: word shape + foreign function words; add a language-ID model (fastText lid.176) if mixed text gets through
    toks = [Token(m.start(), m.end(), m[0]) for m in WORD.finditer(text, start, end)]
    # an elided word before its apostrophe (the ky of ky'okulya) isn't a word to judge
    low = [t.text for t in toks if t.text.islower() and text[t.end:t.end + 1] not in tuple(APOSTROPHES)]
    if not low or FOREIGN.intersection(low) or sum(map(looks_luganda, low)) < 0.6 * len(low):
        return []
    return toks


Writing /tmp/ganda_grammar_bundle/ganda_grammar/rules/base.py


In [9]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/elision.py
"""Elision: a short word ending in a vowel loses it before a vowel-initial word, marked with an apostrophe."""
from .base import APOSTROPHES, Edit, looks_luganda

SRC = "Walusimbi, Empandiika y'Oluganda Entongole (2015)"
RULES = {
    "ELISION_NA": ("na/ne before a vowel: na abantu → n'abantu", f"{SRC}, §1.1.4, §1.3(b)"),
    "ELISION_NGA": ("nga before a vowel: nga ali → ng'ali", f"{SRC}, §1.3(c)"),
    "ELISION_POSS": ("a possessive before a vowel: kya okulya → ky'okulya", f"{SRC}, §1.3(g)"),
    "ELISION_REL": ("bwe, gye in set phrases: bwe atyo → bw'atyo, gye ali → gy'ali", f"{SRC}, §1.3(j)"),
}

# wa is left out: it is also "where?" and stays whole (Ogula wa emmere?, §1.3(r))
TRIGGERS = {"na": "ELISION_NA", "ne": "ELISION_NA", "nga": "ELISION_NGA"} | dict.fromkeys(
    "kya bya lya gwa gya za lwa twa bwa kwa ka ga ya ba".split(), "ELISION_POSS")
NOT_AFTER_POSSESSIVE = {"era", "ate", "oba"}  # conjunctions: a possessive before them ends a phrase
# Only set phrases: bwe, gye, we are also possessives (obulamu bwe, omwana we ali), and those are never elided (§1.3(s))
RELATIVE = {"bwe": {"atyo", "otyo", "etyo", "ati", "oti", "eti"}, "gye": {"ali", "oli"}}


def edits(text, toks):
    out = []
    for a, b in zip(toks, toks[1:]):
        rule = TRIGGERS.get(a.text.lower()) if a.text[1:].islower() else None
        if b.text in RELATIVE.get(a.text.lower(), ()) and a.text[1:].islower():
            rule = "ELISION_REL"
        if (rule and b.text[0] in "aeiou" and len(b.text) > 1 and looks_luganda(b.text)
                and a.end < b.start and not text[a.end:b.start].strip(" \t")
                and not (a.start and text[a.start - 1] in APOSTROPHES + "-")
                and not (rule == "ELISION_POSS" and b.text in NOT_AFTER_POSSESSIVE)):
            out.append(Edit(rule, a.end - 1, b.start, text[a.end - 1:b.start], "'"))
    return out


Writing /tmp/ganda_grammar_bundle/ganda_grammar/rules/elision.py


In [10]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/joining.py
"""Word division: words the official orthography writes as one, and words it writes apart."""
from pathlib import Path

from .agreement import NOUNS
from .base import APOSTROPHES, Edit, looks_luganda

SRC = "Walusimbi, Empandiika y'Oluganda Entongole (2015)"
RULES = {
    "JOIN_NA_PRONOUN": ("na and a pronoun are one word: na bo → nabo, na nze → nange", f"{SRC}, §1.3(a)"),
    "JOIN_FIXED": ("words written as one: buli jjo → bulijjo, mpola mpola → mpolampola, eky'okulabirako → ekyokulabirako",
                   f"{SRC}, §1.3(n), (w), (z)"),
    "SPLIT_NE": ("narrative ne is a word of its own: nebagenda → ne bagenda", f"{SRC}, §1.3(b)"),
    "SPLIT_LOC": ("mu and ku are words of their own: mukibuga → mu kibuga", f"{SRC}, §1.3(d)"),
}

NA_PRONOUN = {"nze": "nange", "ggwe": "naawe", "ye": "naye", "ffe": "naffe", "mmwe": "nammwe", "bo": "nabo",
              "kyo": "nakyo", "byo": "nabyo", "zo": "nazo", "gwo": "nagwo", "gyo": "nagyo", "lyo": "nalyo",
              "bwo": "nabwo", "lwo": "nalwo"}
TWO_WORDS = {  # §1.3(n) bulijjo; §1.3(w) a word said twice is written once
    ("buli", "jjo"): "bulijjo", ("mpola", "mpola"): "mpolampola", ("katono", "tono"): "katonotono",
    ("ddala", "ddala"): "ddaladdala", ("wonna", "wonna"): "wonnawonna", ("okukola", "kola"): "okukolaakola",
    ("okutema", "tema"): "okutemaatema", ("okubuusa", "buusa"): "okubuusabuusa", ("okudaala", "daala"): "okudaaladaala",
}
# §1.3(z) nouns: ekyokulya 'food'. After a class 7/8 word, eky'okulya means 'for eating' (ekintu eky'okulya) and stays.
COMPOUNDS = {"eky": {"okulya", "okulabirako", "okulwanyisa", "okunywa", "okwegezaamu"},
             "eby": {"okulya", "okulabirako", "okulwanyisa", "okunywa", "enjigiriza", "obulamu", "obufuzi"}}
CLASS_7_8 = ("eki", "ebi", "eky", "eby", "ki", "bi")

VERBS = frozenset(line for line in Path(__file__).with_name("verbs.txt").read_text(encoding="utf-8").splitlines()
                  if line and not line.startswith("#"))
# ne + subject prefix + verb stem. Only prefixes whose noun forms start with a or o: n'abagamba sounds nabagamba, so a
# joined nebagamba can only be ne bagamba. Not ki, bi, li, zi, gi: nebikolwa sounds the same as n'ebikolwa.
NE_PREFIXES = ("ba", "tu", "mu", "gu", "ga", "lu", "ka", "bu", "ku")
# The listed nouns without their augment, as they follow mu and ku: ekibuga → kibuga, erinnya → linnya
BARE_NOUNS = frozenset("l" + n[2:] if n.startswith("er") else n[1:] for n in NOUNS)


def _join(first, word):
    return word[0].upper() + word[1:] if first[0].isupper() else word


def edits(text, toks):
    out = []
    for prev, a, b in zip([None] + toks, toks, toks[1:]):
        gap = text[a.end:b.start]
        if not a.text[1:].islower():
            continue
        key = (a.text.lower(), b.text)
        if gap.strip(" \t") == "" and gap and not (a.start and text[a.start - 1] in APOSTROPHES + "-"):
            if key[0] == "na" and b.text in NA_PRONOUN:
                out.append(Edit("JOIN_NA_PRONOUN", a.start, b.end, text[a.start:b.end], _join(a.text, NA_PRONOUN[b.text])))
            elif key in TWO_WORDS:
                out.append(Edit("JOIN_FIXED", a.start, b.end, text[a.start:b.end], _join(a.text, TWO_WORDS[key])))
        elif (len(gap) == 1 and gap in APOSTROPHES and b.text in COMPOUNDS.get(key[0], ())
              and not (prev and not text[prev.end:a.start].strip() and prev.text.lower().startswith(CLASS_7_8))):
            out.append(Edit("JOIN_FIXED", a.end, b.start, gap, ""))
    for t in toks:
        low = t.text[0].lower() + t.text[1:]
        whole = not t.start or not (text[t.start - 1].isalnum() or text[t.start - 1] in APOSTROPHES + "-")
        if not whole or not looks_luganda(low):
            continue
        if low[:2] == "ne" and low[2:4] in NE_PREFIXES and low[4:] in VERBS:
            out.append(Edit("SPLIT_NE", t.start, t.end, t.text, f"{t.text[:2]} {t.text[2:]}"))
        elif low[:2] in ("mu", "ku") and low[2:] in BARE_NOUNS and low[2:] not in VERBS:  # mulinnya 'you climb'
            out.append(Edit("SPLIT_LOC", t.start, t.end, t.text, f"{t.text[:2]} {t.text[2:]}"))
    return out


Writing /tmp/ganda_grammar_bundle/ganda_grammar/rules/joining.py


In [11]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/orthography.py
"""Spelling rules of the official orthography, applied inside single lowercase Luganda words."""
import re
from pathlib import Path

from .base import APOSTROPHES, NC, Edit, looks_luganda

SRC = "Walusimbi, Empandiika y'Oluganda Entongole (2015)"
RULES = {
    "ORTH_L_R": ("r, not l, after e or i: ebilala → ebirala", f"{SRC}, §1.3"),
    "ORTH_VV_NC": ("single vowel before a nasal + consonant (mb, nd, nj, nz…): ensiimbi → ensimbi", f"{SRC}, §1.1.5, §1.1.7"),
    "ORTH_VV_CW": ("single vowel after a consonant + w or y, except ggw and ggy: omukyaala → omukyala", f"{SRC}, §1.1.6"),
    "ORTH_LENGTH": ("long vowels and long consonants as the dictionary spells them: gwanga → ggwanga, omwaana → omwana",
                    f"{SRC}, §1.1.1-1.1.2, §1.2; spellings from the FSI Luganda Basic Course glossary (1968)"),
}
# misspelling → spelling, built by scripts/build_spellings.py
SPELLINGS = dict(line.split("\t")[:2] for line in Path(__file__).with_name("spellings.tsv").read_text(encoding="utf-8").splitlines()
                 if line and not line.startswith("#"))

PATTERNS = (
    ("ORTH_L_R", re.compile(r"(?<=[ei])l"), lambda m: "r"),
    ("ORTH_VV_NC", re.compile(rf"([aeiou])\1(?={NC})"), lambda m: m[1]),
    # Not after ny (one letter, §1.2), ggw/ggy (§1.1.6) or ngw, and not before a final -wo/-yo/-ko/-mu, which
    # lengthens the vowel of a one-syllable verb (§1.1.8: vaawo, faako).
    ("ORTH_VV_CW", re.compile(r"(?<=[bcdfgjklmnprstvzŋ][wy])(?<![gn]g[wy])(?<!ny)([aeiou])\1(?!(?:wo|yo|ko|mu)$)"),
     lambda m: m[1]),
)


def edits(text, toks):
    out = []
    for t in toks:
        after_apostrophe = t.start and text[t.start - 1] in APOSTROPHES
        if not looks_luganda(t.text) or (after_apostrophe and t.start > 1 and text[t.start - 2].isupper()):  # M'okelo
            continue
        found = [Edit(rule, t.start + m.start(), t.start + m.end(), m[0], fix(m))
                 for rule, rx, fix in PATTERNS for m in rx.finditer(t.text)]
        if not found and t.text in SPELLINGS:  # a guide pattern, when one applies, is the better explanation
            found = [Edit("ORTH_LENGTH", t.start, t.end, t.text, SPELLINGS[t.text])]
        out += found
    return out


Writing /tmp/ganda_grammar_bundle/ganda_grammar/rules/orthography.py


In [12]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/spellings.tsv
# Misspellings of vowel length and doubled consonants (Walusimbi 2015, §1.1.1-1.1.2, §1.2), built by
# scripts/build_spellings.py from the FSI Luganda Basic Course glossary (1968) and corpus counts.
# misspelling	spelling	count of the misspelling	count of the spelling
gwanga	ggwanga	906	12210
olunnaku	olunaku	404	5671
nnga	nga	289	365404
lunnaku	lunaku	263	9099
egwanga	eggwanga	259	5797
omwaana	omwana	222	14500
dala	ddala	194	20843
emyaaka	emyaka	181	13236
polisi	poliisi	169	29968
myaaka	myaka	153	8775
somero	ssomero	137	4559
dwaliro	ddwaliro	89	2299
mwaana	mwana	87	5048
byaalo	byalo	84	2039
nnaye	naye	78	56625
wanno	wano	78	9221
edakiika	eddakiika	70	790
emyeezi	emyezi	69	2658
maso	maaso	66	20872
esomero	essomero	59	1613
nimiro	nnimiro	59	1614
ennyonnyi	ennyonyi	58	1076
lero	leero	58	6798
eddakika	eddakiika	55	790
enyonyi	ennyonyi	55	1076
akawungezi	akawungeezi	54	1559
kkoti	kkooti	53	17285
nyonyi	nnyonyi	46	749
ebaluwa	ebbaluwa	43	2279
edagala	eddagala	43	4156
engato	engatto	43	931
ensiimbi	ensimbi	41	6992
sebo	ssebo	41	483
kibina	kibiina	39	7101
okuda	okudda	38	5195
okuja	okujja	37	2791
dagala	ddagala	35	1624
enviri	enviiri	35	1074
amaso	amaaso	34	3671
ekibina	ekibiina	34	5715
ddakika	ddakiika	33	571
myeezi	myezi	33	1588
ddwaliiro	ddwaliro	32	2299
ebyaalo	ebyalo	32	523
emitwaalo	emitwalo	32	2240
obukade	obukadde	30	7792
ccaayi	caayi	27	342
naaye	naye	26	56625
emmeere	emmere	25	5160
mitwaalo	mitwalo	25	624
nnyonnyi	nnyonyi	24	749
omusubuzi	omusuubuzi	24	1016
kuja	kujja	23	1258
omusaanvu	omusanvu	23	1164
biffo	bifo	21	4185
kubuza	kubuuza	21	245
ngato	ngatto	21	289
mukaga	mukaaga	20	3293
ebibina	ebibiina	19	1848
edwaliro	eddwaliro	19	677
ennyanya	ennyaanya	19	317
ebikuumi	ebikumi	18	647
mmeere	mmere	18	2187
mmottoka	mmotoka	17	8621
mmotokka	mmotoka	16	8621
enimiro	ennimiro	15	758
kkuumi	kkumi	15	2352
nsiimbi	nsimbi	15	2314
olukiko	olukiiko	15	2725
emmottoka	emmotoka	14	4203
omupira	omupiira	14	2946
ssaaza	ssaza	14	1420
eterekero	etterekero	13	162
kibuuga	kibuga	13	7523
mmeza	mmeeza	13	570
musubuzi	musuubuzi	13	609
okwoogera	okwogera	13	4041
omukaga	omukaaga	13	627
buuli	buli	12	48306
eddaggala	eddagala	12	4156
kibugga	kibuga	12	7523
omuliiro	omuliro	12	4105
tuntu	ttuntu	12	366
engeeri	engeri	11	9213
ggali	ggaali	11	211
kibbi	kibi	11	1695
kutekateeka	kuteekateeka	11	170
lukiko	lukiiko	11	4064
omunyo	omunnyo	11	302
waano	wano	11	9221
eddaakiika	eddakiika	10	790
eggali	eggaali	10	268
esaza	essaza	10	538
kitunddu	kitundu	10	8891
kunaba	kunaaba	10	187
kwoogera	kwogera	10	1832
nnze	nze	10	10553
olugi	oluggi	10	649
akakade	akakadde	9	611
eddwaliiro	eddwaliro	9	677
kulowoza	kulowooza	9	644
kumannya	kumanya	9	1395
luda	ludda	9	3067
nyinimu	nnyinimu	9	108
okulaaba	okulaba	9	11717
okuleeka	okuleka	9	606
okusiinziira	okusinziira	9	4296
abakade	abakadde	8	1086
bbaaluwa	bbaluwa	8	1027
ddaggala	ddagala	8	1624
egulo	eggulo	8	2349
ekikoopo	ekikopo	8	1014
eriso	eriiso	8	528
nnyaanja	nnyanja	8	2363
obuungi	obungi	8	3178
okuwuulira	okuwulira	8	2338
omutwaalo	omutwalo	8	236
ppoliisi	poliisi	8	29968
bbuli	buli	7	48306
bibina	bibiina	7	1229
byenyanja	byennyanja	7	133
ddaakiika	ddakiika	7	571
ebifaanannyi	ebifaananyi	7	2043
ebiffo	ebifo	7	2715
emipira	emipiira	7	1390
essowani	essowaani	7	71
kikuumi	kikumi	7	1162
kubbala	kubala	7	371
kudduuka	kudduka	7	223
lugi	luggi	7	397
mmotooka	mmotoka	7	8621
okwogeera	okwogera	7	4041
oluvannyuuma	oluvannyuma	7	17949
oluvvannyuma	oluvannyuma	7	17949
ssazza	ssaza	7	1420
terekero	tterekero	7	108
ttaaba	taaba	7	98
amafuuta	amafuta	6	2470
bitabbo	bitabo	6	801
ebitabbo	ebitabo	6	1587
ekkalamu	ekkalaamu	6	68
emmeza	emmeeza	6	300
kifaanaanyi	kifaananyi	6	854
kiibiina	kibiina	6	7101
kiiki	kiki	6	2273
kulisa	kuliisa	6	72
kwogeera	kwogera	6	1832
mmaaso	maaso	6	20872
muga	mugga	6	691
ngaa	nga	6	365404
ofisi	ofiisi	6	4484
okugeenda	okugenda	6	8445
okutunuula	okutunula	6	291
okwawuula	okwawula	6	427
ooba	oba	6	48600
akawuungeezi	akawungeezi	5	1559
bbaluuwa	bbaluwa	5	1027
buluungi	bulungi	5	16085
buungi	bungi	5	2894
dirisa	ddirisa	5	224
ebituundu	ebitundu	5	4641
ekibugga	ekibuga	5	3189
ekitongoole	ekitongole	5	4751
ekolero	ekkolero	5	259
embizi	embizzi	5	626
essaaza	essaza	5	538
gulo	ggulo	5	190
kituundu	kitundu	5	8891
kutusa	kutuusa	5	252
luguuddo	luguudo	5	3409
lwokutano	lwokutaano	5	109
mambuuka	mambuka	5	158
mulwade	mulwadde	5	1022
mupira	mupiira	5	1753
ngeeri	ngeri	5	15256
nviri	nviiri	5	349
okutuunda	okutunda	5	1477
okuziimba	okuzimba	5	3278
okwagaala	okwagala	5	3739
amalalo	amalaalo	4	121
bituundu	bitundu	4	9009
buggwanjuba	bugwanjuba	4	231
bukade	bukadde	4	2701
ebifaanaanyi	ebifaananyi	4	2043
efumbiro	effumbiro	4	41
ekibuuga	ekibuga	4	3189
ekifaanaanyi	ekifaananyi	4	1648
ekifaanannyi	ekifaananyi	4	1648
ennyaanja	ennyanja	4	852
enyaanya	ennyaanya	4	317
ffee	ffe	4	4711
ggwannga	ggwanga	4	12210
kakade	kakadde	4	327
kikkumi	kikumi	4	1162
kkulya	kulya	4	1157
kulaaba	kulaba	4	2335
kummanya	kumanya	4	1395
kutuunda	kutunda	4	601
kwagaala	kwagala	4	1570
mafuuta	mafuta	4	603
mutwaalo	mutwalo	4	74
muzzadde	muzadde	4	368
nngeri	ngeri	4	15256
nyaanya	nnyaanya	4	56
obuggaali	obugaali	4	141
obunnene	obunene	4	1119
okukebeera	okukebera	4	552
okummala	okumala	4	6449
okusittula	okusitula	4	560
oluda	oludda	4	1508
omuwaala	omuwala	4	4003
wakatti	wakati	4	15060
amaawanga	amawanga	3	2417
bakade	bakadde	3	1099
bbaanga	bbanga	3	3191
bbasi	bbaasi	3	1354
biibiina	bibiina	3	1229
bitaabo	bitabo	3	801
bitunddu	bitundu	3	9009
ebbaluuwa	ebbaluwa	3	2279
ebitoongole	ebitongole	3	1935
ekikoppo	ekikopo	3	1014
ekitonngole	ekitongole	3	4751
emmotooka	emmotoka	3	4203
emyakka	emyaka	3	13236
endiiga	endiga	3	965
enjawuulo	enjawulo	3	1012
erinyo	erinnyo	3	65
kiibuga	kibuga	3	7523
kikoopo	kikopo	3	521
kuda	kudda	3	1101
kukkula	kukula	3	188
kukyussa	kukyusa	3	595
kulundda	kulunda	3	176
kuteekateka	kuteekateeka	3	170
kuwoonya	kuwonya	3	87
mmootoka	mmotoka	3	8621
munyo	munnyo	3	96
musaawo	musawo	3	564
mwakka	mwaka	3	8356
myezzi	myezi	3	1588
neda	nedda	3	784
ngga	nga	3	365404
nnimiiro	nnimiro	3	1614
nnyinnimu	nnyinimu	3	108
nsaalo	nsalo	3	932
okkubuuza	okubuuza	3	454
okubaala	okubala	3	644
okubeeraa	okubeera	3	10738
okufunna	okufuna	3	12174
okugoloola	okugolola	3	121
okukyussa	okukyusa	3	1900
okulabiika	okulabika	3	564
okumaanya	okumanya	3	5692
okunnywa	okunywa	3	1119
okussanga	okusanga	3	486
okuttunda	okutunda	3	1477
okuvuunda	okuvunda	3	165
okuyaamba	okuyamba	3	3128
olunakku	olunaku	3	5671
omuga	omugga	3	428
omusabaze	omusaabaze	3	91
omuzade	omuzadde	3	564
omweenge	omwenge	3	1544
poliissi	poliisi	3	29968
ssebbo	ssebo	3	483
tumbi	ttumbi	3	185
waakati	wakati	3	15060
wakkati	wakati	3	15060
akakaadde	akakadde	2	611
amafutaa	amafuta	2	2470
amafutta	amafuta	2	2470
amaziima	amazima	2	2053
bambii	bambi	2	381
bifaanaanyi	bifaananyi	2	454
biffaananyi	bifaananyi	2	454
ebbaaluwa	ebbaluwa	2	2279
ebiitabo	ebitabo	2	1587
ebikoopo	ebikopo	2	363
ebiteeteyi	ebiteeteeyi	2	78
ebitunddu	ebitundu	2	4641
eddaagala	eddagala	2	4156
eddagalaa	eddagala	2	4156
eekitabo	ekitabo	2	1121
eggwannga	eggwanga	2	5797
ekitaabo	ekitabo	2	1121
ekituundu	ekitundu	2	3931
ekkibiina	ekibiina	2	5715
emisannvu	emisanvu	2	205
emmotokka	emmotoka	2	4203
empeeta	empeta	2	443
endigga	endiga	2	965
engaatto	engatto	2	931
esanduuko	essanduuko	2	432
etteerekero	etterekero	2	162
ggwangga	ggwanga	2	12210
kabadda	kabada	2	46
kasirivu	kasiriivu	2	50
katonno	katono	2	2386
kawungezi	kawungeezi	2	153
kawuungeezi	kawungeezi	2	153
kibii	kibi	2	1695
kiifo	kifo	2	12593
kiirungi	kirungi	2	2839
kkunywa	kunywa	2	384
kkuyita	kuyita	2	708
kolero	kkolero	2	152
kuffuna	kufuna	2	2839
kugeenda	kugenda	2	3392
kukyaalira	kukyalira	2	69
kuleeka	kuleka	2	214
kumaanya	kumanya	2	1395
kusiinga	kusinga	2	281
kusittula	kusitula	2	181
kusituula	kusitula	2	181
kussanyusa	kusanyusa	2	196
kutaamiira	kutamiira	2	20
kutaandika	kutandika	2	1419
kutunnula	kutunula	2	202
kutuunga	kutunga	2	29
kwagalaa	kwagala	2	1570
lukkumi	lukumi	2	442
lukuumi	lukumi	2	442
lunakku	lunaku	2	9099
maserengetta	maserengeta	2	119
maziima	mazima	2	4018
mbizi	mbizzi	2	218
mmawanga	mawanga	2	3409
mmuzadde	muzadde	2	368
mpetta	mpeta	2	141
mpiiso	mpiso	2	66
mulwaadde	mulwadde	2	1022
musuubuuzi	musuubuzi	2	609
muuntu	muntu	2	12403
mwanna	mwana	2	5048
ndigga	ndiga	2	310
njawuulo	njawulo	2	1616
nkooko	nkoko	2	477
nnsalo	nsalo	2	932
nnyannja	nnyanja	2	2363
obukkadde	obukadde	2	7792
obuwaanvu	obuwanvu	2	905
okkudda	okudda	2	5195
okkugamba	okugamba	2	1346
okkulaba	okulaba	2	11717
okkumanya	okumanya	2	5692
okubaja	okubajja	2	97
okubbala	okubala	2	644
okubbeera	okubeera	2	10738
okufuuga	okufuga	2	1046
okulisa	okuliisa	2	183
okusibba	okusiba	2	626
okusirikka	okusirika	2	140
okusomesaa	okusomesa	2	1204
okussiba	okusiba	2	626
okuttonnya	okutonnya	2	232
okuttuusa	okutuusa	2	10667
okutuussa	okutuusa	2	10667
okuvvunda	okuvunda	2	165
okuwuliira	okuwulira	2	2338
okuyiita	okuyita	2	3304
okwozza	okwoza	2	201
olugguudo	oluguudo	2	1498
olukuumi	olukumi	2	147
oluvannyumma	oluvannyuma	2	17949
olwokusattu	olwokusatu	2	125
olwomukaga	olwomukaaga	2	25
olwookubiri	olwokubiri	2	229
omucceere	omuceere	2	223
omusaawo	omusawo	2	1474
omussawo	omusawo	2	1474
omusuubuuzi	omusuubuzi	2	1016
omwakka	omwaka	2	6197
ookudda	okudda	2	5195
sinnga	singa	2	8846
ttunttu	ttuntu	2	366
wakaati	wakati	2	15060
walaa	wala	2	1766


Writing /tmp/ganda_grammar_bundle/ganda_grammar/rules/spellings.tsv


In [13]:
%%writefile /tmp/ganda_grammar_bundle/ganda_grammar/rules/verbs.txt
# Verb stems (the infinitive without oku-), used to tell a verb from other words.
# Infinitives seen at least 10 times in the first 60,000 rows of henryhale/luganda-pretrain2 without tinystories-ganda,
# plus the infinitives of the English-Luganda dictionary (gandalm/resources/English-Luganda_version_2_0.txt).
# Consonant-initial stems only. Some are misspelled as written in the corpus; they only make a word count as a verb.
baaga
baagala
baagazisa
baaniriza
baawula
babala
babangula
babanja
babba
babeera
babeerera
babonereza
babonyaabonya
babudaabuda
babula
babuliira
babulira
babuulira
babuulirira
babuuza
babuzaabuza
baddiza
baddukirira
baddusa
badduukirira
bafera
bafiiriza
bafuga
bafulumya
bafumbira
bafuna
bafunira
bafuula
baga
bagaana
bagabira
bagabula
bagamba
baganya
bagasa
bagatta
bagema
bagezesa
baggalira
baggulira
baggya
bagoba
bagobaganya
bagoberera
bagula
bagulira
bagumbulula
bagumya
bagunjula
baguza
bagwa
bajja
bajjanjaba
bajjira
bajjukira
bajjukiza
baka
bakaka
bakakasa
bakaluubirira
bakaluubiriza
bakangavvula
bakebera
bakema
bakiikirira
bakiriza
bakkakkanya
bakkiriza
bakola
bakolanga
bakolera
bakoppa
bakosa
bakozesa
bakuba
bakubagiza
bakubira
bakubiriza
bakulaakulanya
bakulembera
bakunya
bakuuma
bakuumira
bakuza
bakwana
bakwasa
bakwata
bakwatira
bakyalira
bakyawa
bakyusa
bala
balaba
balabirira
balabula
balaga
balagira
balagirira
balambika
balambula
balamula
balanga
balangira
balangula
balanguza
baleeta
baleetera
baleka
balekera
balema
balemesa
balibwa
baliga
baliisa
balimba
balinnya
balirira
baliyirira
baloga
balokola
balonda
balondoola
balongoosa
baloopa
baluka
balula
baluma
balumba
balumiriza
balungamya
balwanirira
balwanyisa
balwaza
balya
bamanya
bamanyisa
bamatiza
bambaza
bamenyera
bamma
bamwa
banaaza
banga
banguka
bangula
bangulwa
banguyiza
banja
banjulira
bannyonnyola
banoonya
banoonyeza
banunula
banyaga
banyiga
banyigiriza
banyonyola
banyweza
bapimira
basaasira
basaba
basabira
basakira
basala
basalira
basaliriza
basanga
basanyusa
basasula
basasulira
basasuza
basembeza
basendasenda
basengula
basiba
basibira
basiibula
basiiga
basiima
basikiriza
basindiikiriza
basindika
basindikira
basinga
basingisa
basisinkana
basobozesa
basomera
basomesa
basonyiwa
basosola
bassa
basumulula
basunsula
basuubiza
basuula
basuulirira
basuuza
basuza
baswaza
bata
bataasa
batabaganya
batabula
batambuza
batangira
batangirira
batawaanya
bateega
bateeka
bateekateeka
bateekerateekera
bateera
bategeera
bategeeza
bategekera
batema
batendeka
batigomya
batiisa
batiisatiisa
batikka
batiza
batizibwa
batta
battira
batulugunya
batumira
batunda
batuukirira
batuusa
batuuza
batwaala
batwala
batwalira
baviira
bavuma
bavunaana
bawa
bawabula
bawagira
bawaliriza
bawamba
bawandiika
bawandiikira
bawandiisa
bawanga
bawangula
bawasa
baweebwa
baweerera
baweereza
baweerezanga
bawemula
bawola
bawolereza
bawondera
bawonya
bawulira
bawuliriza
bawuliza
bayamba
bayigga
bayigiriza
bayiira
bayimbula
bayimiriza
bayingirira
bayingiza
bayisa
bayita
baza
bazaala
bazaalira
baziika
bazikiriza
bazimba
bazimbira
bazinda
baziyiza
bazuula
bazza
bba
bbibwa
bbika
bbinkana
bbira
bbirira
bbisa
bbula
bbulula
beebaza
beefuulira
beegayirira
beekebejja
beekengera
beela
beera
beeranga
beerera
beerwa
beesiga
beetegereza
beewala
beewanika
beewuunyisa
beeyimirira
bega
bejjagala
bejjerera
bela
bembereza
bera
betenta
biddukanya
bifulumya
bifuna
bifunira
bifuula
biggya
bigoberera
bigonjoola
bigula
biika
bijja
bika
bikebera
bikira
bikirira
bikka
bikkirira
bikkiriza
bikkula
bikkulibwa
bikkulirwa
bikola
bikolanga
bikolera
bikozesa
bikula
bikulaakulanya
bikulirwa
bikuuma
bikwata
bikyusa
bimanya
binda
bindabinda
binika
bira
biraba
biraga
bireeta
bireka
birima
biriza
birizibwa
birongoosa
birwanyisa
birya
bisa
bisala
bisasulira
bisibwa
bisimba
bisiwaza
bisobozesa
bisoma
bissa
bitebya
biteeka
bitegeera
bitegeeza
bitereeza
bitereka
bitunda
bituukiriza
bituusa
bitwala
bivvuunuka
biwa
biwuliriza
biyiga
biyigiriza
biyingiza
biyita
biyonja
bizza
bobbya
boggola
bonaabona
bonabona
boneka
bonereza
bonerezebwa
bongera
bongeza
bonyaabonya
bonyaabonyezebwa
bonyabonyezebwa
booga
boogerera
bookya
boola
boolekera
botola
bowa
budaabuda
budaabudibwa
budabuda
budabudibwa
budamya
bufuna
buggya
bugiriza
bugonjoola
buguma
bugumiriza
bugumya
buguumirira
buguutanya
buka
bukala
bukola
bukozesa
bukuuma
bukwasa
bula
bulaba
bulankanya
bulira
bulirira
buliriza
bulwa
bulwanyisa
bumalira
bumba
bumbujja
bumbujjira
bumbulukuka
buna
bunya
bunyisa
bunyisibwa
bunyweza
busaasaanya
busabusa
busiiga
busoma
buteeka
butuka
butuukiriza
butuusa
butwala
buuka
buukabuuka
buukira
buulira
buuliranga
buulirira
buulirirwa
buuliriza
buulirwa
buumba
buusa
buusabuusa
buuza
buuzibwa
buvvuunuka
buwa
buyingira
buza
buzaabuza
buzaabuzibwa
buzabuza
buzibwa
buzibwabuzibwa
buzza
bwatuka
bwatukira
bwebwena
bwewala
bweyambisa
byambala
byamuka
byangatana
byekebejja
byekolera
byesonyiwa
byetegereza
byettanira
byewala
byeyambisa
byogera
caaka
cakala
cakalira
camuka
camula
camuukirira
canga
cankalana
cankalanya
cupula
daabiriza
daala
dabiriza
dda
ddaabiriza
ddaabirizibwa
ddaabulula
ddabiriza
ddanga
ddingana
ddira
ddirira
ddiriza
ddiza
ddizibwa
ddiŋŋana
dduka
ddukana
ddukannya
ddukanya
ddukanyizibwa
ddukira
ddukiranga
ddukirira
ddusa
dduukira
dduukirira
diba
dibaga
dibuuda
dibuula
dibya
digida
diibuuda
dobonkana
dobonkanya
duka
dukanya
duukirira
duula
duumira
faafaagana
faafagana
faanagana
faanana
faananyizibwa
fafaagana
fafagana
fafaganya
fanana
fananya
feebya
feffetta
fera
ffa
ffuna
fiira
fiirira
fiiriza
fiirizibwa
fiirwa
firwa
fissa
fuba
fubutuka
fudemba
fufuggaza
fuga
fuganga
fugibwa
fugira
fuka
fukamira
fukirira
fukumuka
fula
fuluma
fulumira
fulumizibwa
fulumya
fuluuta
fuma
fumba
fumbira
fumbiriganwa
fumbirwa
fumbisa
fumbiza
fumiitiriza
fumintiriza
fumita
fumitibwa
fumitiriza
funa
funanga
funda
fundikira
fundikirwa
fungiza
funibwa
funika
funira
funirwa
funyisa
funyulula
funza
futiza
futyanka
fuuka
fuukafuuka
fuukira
fuukuuka
fuula
fuulibwa
fuuna
fuuwa
fuuweeta
fuuyira
fuuyirwa
fuuyisa
fuuza
fuyira
fuyisa
gaana
gaanibwa
gaanira
gaaya
gaba
gabana
gabanibwa
gabanya
gabanyizibwa
gabibwa
gabira
gabirira
gabirwa
gabula
gabulwa
gabwa
gafuna
gafuula
gaggawala
gaggawaza
gaggya
gagula
gajambula
gajambulwa
gakima
gakola
gakozesa
gakuuma
galamira
galamiza
galamizibwa
gamba
gambibwa
gana
gandawaza
ganja
ganyula
ganyulwa
ganyurwa
ganza
gasa
gasimbagana
gassa
gata
gatambuza
gateeka
gatibwa
gatta
gattibwa
gattika
gattira
gattulula
gattululwa
gatunda
gatuusa
gatwala
gawa
gayaala
gayaalirira
gazimba
gaziwa
gaziya
gaziyizibwa
gazza
gejja
gema
gemebwa
gemesa
gemwa
genda
gendanga
gendda
gendera
genderera
gengewala
genyiwala
genza
gera
geraageranya
geraageranyizibwa
gerageranya
gereesa
gereka
geya
geza
gezebwa
gezesa
gezesebwa
gezza
ggala
ggalira
ggalirwa
ggalwa
ggibwa
ggoberera
gguka
ggula
ggulwa
ggumira
ggumiza
ggusa
ggwa
ggweera
ggya
ggyibwa
gibawa
gibba
gibwa
giddaabiriza
giddukanya
gifulumya
gifuna
gifuula
gigaana
giggala
giggya
gigoba
gigoberera
gigula
gikanika
gikebera
gikola
gikolera
gikozesa
gikuba
gikuuma
gikuwa
gikwasa
gikwata
gikyusa
gimaliriza
gimanya
gimusa
gimuwa
ginoonya
ginunula
gipangisa
gira
giraba
girabirira
giraga
gireeta
gireka
giremesa
girima
girondoola
girongoosa
girumba
girwanyisa
girya
gisaba
gisanga
gisasula
gisiba
gisiiga
gisimba
gisoma
gissa
gisuula
gitambuza
giteeka
gitegeera
gitegeeza
gitereeza
gitunda
gituukiriza
gituusa
gituwa
gitwala
givuga
giwa
giwagira
giwandiika
giwangula
giwulira
giyamba
giyingira
giyingiza
giyisa
giyita
gizimba
gizza
goba
gobaganya
gobaganyizibwa
gobebwa
gobelera
gobera
goberera
gobereranga
gobererwa
gobolola
gobwa
gogola
golokoka
golokosa
golola
gonda
gondera
gonderanga
gonjola
gonjoola
gonjoolwa
gonza
goomya
gootaana
gootaanya
gotaanya
gubawa
gufuna
gufuula
guggya
gugoba
gugulana
gugumuka
gukebera
gukola
gukozesa
gukuuma
gukwata
gula
gulaba
gulanga
guleka
gulibwa
gulira
gulirira
gulumba
gulumiza
gulumizibwa
gulwa
gulwanyisa
gulya
guma
gumaliriza
gumba
gumbulula
gumiikiriza
gumikiriza
guminkiriza
gumira
gumya
gunja
gunjula
gunywa
gusasula
gussa
gutangira
gutuusa
gutwala
guumaaza
guwa
guwangula
guwulira
guyita
guza
guzannya
guziika
guzikiza
guzza
gwa
gwekebejja
gwira
gya
gyambala
gyawula
gyekebejja
gyekenneenya
gyemera
gyetegereza
gyetooloola
gyettanira
gyeyambisa
gyibwa
gyoza
jaganya
jaguliza
jaguza
janjaba
janjabibwa
janjabwa
jeema
jeemera
jelega
jimusa
jingajinga
jingirira
jja
jjaguza
jjanga
jjanjaba
jjanjabibwa
jjanjabirira
jjanjabirwa
jjanjabwa
jjibwa
jjira
jjukira
jjukiranga
jjukirwa
jjukiza
jjukizanga
jjula
jjulira
jjulukuka
jjulula
jjumbira
jjumbirwa
jjuza
jjuzibwa
jojobya
jojolikana
jolonga
jooga
joogebwa
joonyesa
jugumira
jukira
jula
julira
juliza
julukuka
jumbira
juna
juza
kaaba
kaabira
kaabirira
kaabya
kaaniika
kaanya
kaatiriza
kaayana
kaayanira
kabala
kabasanya
kabassanya
kabawaza
kaddiwa
kafuna
kaka
kakalabya
kakana
kakanya
kakanyala
kakasa
kakasibwa
kakasiddwa
kakasiza
kakata
kakkalabya
kakkana
kakkanya
kakkasa
kakozesa
kakuba
kakuuma
kakwata
kala
kalaba
kalakata
kaliga
kalilira
kalira
kalirira
kaluba
kalubiriza
kalubya
kaluubirira
kaluubirirwa
kaluubiriza
kaluubirizibwa
kama
kamala
kambuwala
kamula
kanda
kanga
kangavula
kangavvula
kangavvulwa
kangisa
kangula
kanika
kankana
kankanya
kansa
kanyuga
kasa
kasuka
kasukira
kavvula
kaza
kazimba
kebela
kebera
keberanga
keberebwa
keberwa
kebeza
kedenza
keera
keeranga
keerewa
keesa
keewa
kekkereza
kema
kemebwa
kendeera
kendeeza
kendeezebwa
kendera
kendeza
kenena
kenenula
kenga
kenula
kerewa
ketta
kifuna
kifuula
kifuuwa
kiggumiza
kiggya
kigula
kigumira
kiika
kiikirira
kiikirirwa
kiina
kijja
kijjanya
kijjukira
kikakasa
kikiitana
kikirira
kikkaatiriza
kikkiriza
kikola
kikolera
kikolerera
kikomya
kikozesa
kikulembera
kikulira
kikuuma
kikwata
kikyusa
kiliza
kima
kimaliriza
kimanya
kimanyiira
kimuwa
kinnimba
kinnyonnyola
kinogaanya
kinogera
kinunula
kira
kiraba
kirabisa
kiraga
kirambula
kireeta
kireka
kiriza
kirizibwa
kiriziganya
kirwanyisa
kirya
kisa
kisala
kisalira
kisanga
kisasulira
kisima
kisoma
kissa
kiteeka
kitegeera
kitegeerera
kitereeza
kitunda
kitunuulira
kituukiriza
kitwala
kiwa
kiwagira
kiwakanya
kiwamba
kiwandiika
kiwandiisa
kiwangula
kiwulira
kiyiga
kiyingira
kiyisa
kiyita
kizikiriza
kizimba
kizuula
kizza
kka
kkaanya
kkaatiriza
kkakana
kkakanya
kkakkana
kkakkanya
kkalira
kkanya
kkekkereza
kkira
kkirira
kkiririza
kkiriza
kkirizibwa
kkiriziganya
kkirizisa
kkirizza
kkusa
kkuta
koboza
kogga
kola
kolagana
kolaganira
kolanga
kolebwa
koleddwa
koleeza
kolera
kolerera
kolerwa
kolima
kolimira
kolobola
kolokota
kolola
kolwa
koma
komaga
komba
komekereza
komekerezebwa
komekkereza
komekkerezebwa
komerera
komererwa
komezebwa
kommonta
komola
komolebwa
komolwa
komya
kona
konagaga
kongojja
kongoola
konjera
konkona
kontana
konziba
koobera
kookoonya
koola
koona
koonagana
koowa
koowoola
kooya
kopa
koppa
kosa
kosebwa
koseza
kotoggera
kowoola
kozesa
kozesanga
kozesebwa
kuba
kubagana
kubaganya
kubagiza
kubba
kubibwa
kubira
kubiriza
kubisa
kubudaabuda
kubuulira
kubuuza
kubwa
kuddiza
kufiiriza
kufuga
kufunira
kufuula
kugaggawaza
kugamba
kuganza
kugasa
kuggya
kugira
kugirwa
kugoba
kuguka
kuguza
kujjukira
kukakasa
kukebera
kukkiriza
kukola
kukolera
kukozesa
kukuba
kukubira
kukula
kukulembera
kukusa
kukuta
kukuuma
kukwata
kukyawa
kula
kulaakulana
kulaakulanya
kulaakulanyizibwa
kulaba
kulabirira
kulabisa
kulaga
kulakulana
kulakulanya
kuleetera
kuleka
kulema
kulembera
kulemberwa
kulembeza
kulemesa
kulira
kuliza
kulokola
kulongoosa
kulugguka
kulukusa
kulukuta
kulula
kuluma
kulumba
kulumya
kulwanyisa
kulwaza
kuma
kumaakuma
kumakuma
kumanya
kumanyisa
kumba
kumibwa
kumpanya
kunama
kunga
kungaana
kungaanira
kungaanya
kunganya
kungubaga
kungubagira
kungula
kungulwa
kunnyonnyola
kunoonya
kunta
kunya
kunyiiza
kunyumira
kunyumiza
kusa
kusaba
kusabira
kusanga
kusanyusa
kusasula
kusiba
kusinga
kusisinkana
kusobozesa
kusonyiwa
kusuula
kuta
kutaasa
kutama
kuteeka
kutegeera
kutegeeza
kutendereza
kutta
kutuka
kutula
kutulakutula
kutuukirira
kutuusa
kutwala
kutwalira
kuubagana
kuula
kuuma
kuumanga
kuumibwa
kuumira
kuumirwa
kuumwa
kuusakuusa
kuuta
kuutira
kuvuma
kuwa
kuwandiikira
kuweereza
kuwonya
kuwulira
kuwuliriza
kuyamba
kuyingira
kuyisa
kuyita
kuyunga
kuza
kuzibuwalira
kuzibwa
kuzikiriza
kuziyiza
kuŋŋaana
kuŋŋaanya
kwaana
kwaata
kwaatagana
kwagala
kwajja
kwana
kwanaganya
kwasa
kwasaganya
kwasibwa
kwasisa
kwasiza
kwata
kwatagana
kwataganira
kwataganya
kwatanga
kwatibwa
kwatira
kwatirira
kwatirwa
kwattibwa
kwebaza
kweeka
kweka
kwekeeta
kwekengera
kwekula
kwesiga
kwetaagisa
kwewuunyisa
kwongera
kya
kyaala
kyaawa
kyagenda
kyakira
kyala
kyalira
kyama
kyambala
kyamuka
kyamula
kyamya
kyankalana
kyankalanya
kyawa
kyawagana
kyayibwa
kyayisa
kyaza
kyebungulula
kyekenneenya
kyerabira
kyetegereza
kyetooloola
kyewala
kyima
kyogera
kyoleka
kyuka
kyukakyuka
kyukira
kyupula
kyusa
kyusakyusa
kyusibwa
kyusiza
kyuuka
kyuusa
laajana
laajanira
laakulana
laakulanya
laalika
laama
laawa
laaza
laba
labagana
labanga
labankana
labibwa
labika
labikira
labikirwa
labilira
labira
labirira
labiriranga
labirirwa
labiriza
labirwa
labisa
labisibwa
labula
labulwa
labwa
lafuubana
laga
lagaana
lagajjalira
lagibwa
lagira
lagirira
lagirwa
lagula
lagulwa
lakira
lakulana
lakulanya
lala
lalukira
lamaga
lamba
lambaala
lambika
lambikibwa
lambula
lambulanga
lambulula
lambuula
lambuza
lamula
lamulwa
lamusa
lamuza
landa
langa
langibwa
langira
langirira
langirirwa
lanya
lasa
lawuna
layira
layiza
layizibwa
lebeesa
lebeeta
lebera
lebeza
leega
leekaana
leekaanira
leeta
leetanga
leetebwa
leetera
lega
legejja
legesa
leguula
leka
lekaana
lekanga
lekanira
lekebwa
lekera
lekulira
lema
lemagana
lemala
lemba
lembeka
lembera
lemera
lemererwa
lemesa
lemesebwa
lemeza
lemwa
lengera
lengezza
lera
leta
letebwa
letera
libba
liddukanya
lifuna
liggya
ligula
liguza
liibwa
liikiriza
liira
liisa
likola
likozesa
likuba
likulaakulanya
likuuma
lilanya
lima
limba
limbalimba
limbibwa
limibwa
limira
limirira
limwa
lina
linaana
linda
lindanga
lindirira
linga
lingiza
linnya
linnyira
linnyirira
linnyirirwa
linnyisa
linnyisibwa
linya
linyirira
linyisa
linyisibwa
lippa
lira
liraana
lirana
lissa
liteeka
litegeera
litunda
litwala
liwa
liyirira
liyirirwa
liyita
lizza
loga
loka
lokoka
lokola
lokolebwa
lomba
londa
londebwa
londera
londerera
londesa
londola
londoola
londoolwa
longooka
longoosa
longoosebwa
longosa
longosebwa
lonkoma
loonda
loondola
loopa
loota
lowooleza
lowooza
lowoozanga
lowoozesa
lowoza
loza
lufuula
luka
lukola
lukozesa
lukuba
luluka
lulunkanira
luma
lumba
lumbagana
lumbibwa
lumika
lumirirwa
lumiriza
lumirizibwa
lumirwa
lumizibwa
lumwa
lumya
lunda
lundibwa
lundira
lungamizibwa
lungamya
lungiya
lusoma
lutwala
luubirira
luutinga
luŋŋamya
lwa
lwaala
lwaana
lwaanirira
lwaanyisa
lwala
lwalalwala
lwana
lwanagana
lwanira
lwanirira
lwanisa
lwanyisa
lwanyisibwa
lwaza
lya
lyamuka
lyanga
lyowa
maamira
magamaga
mala
malira
malirira
maliriza
malirizibwa
mannya
mannyikwa
mansa
manya
manyagana
manyanga
manyibwa
manyiddwa
manyiira
manyiiza
manyika
manyira
manyisa
manyisibwa
masamasa
matira
matiza
matizibwa
mbudaabuda
mbuulira
mbuuza
megenya
megga
meggana
meketa
menya
menyebwa
menyeka
menyekamenyeka
mera
meremeenya
meruka
merusa
meza
mila
mira
mma
mmanya
mmuka
mpa
mpanga
mpasa
mpeerera
mpeereza
mpisa
mpita
mpolereza
mponya
mpulira
mpuliriza
mubanja
mubatiza
mubba
mubbira
mubeera
mubikira
muboggolera
mubonereza
mubudaabuda
mubuulira
mubuulirira
mubuuza
mubuzaabuza
mucamula
muddira
muddiza
muddusa
mudduukirira
mufaanana
mufera
mufiiriza
mufulumya
mufumbirwa
mufumbiza
mufumita
mufuna
mufunira
mufunyisa
mufuula
mufuuwa
mufuuyira
mugaana
mugajambula
mugamba
muganza
mugatta
mugeraageranya
mugezesa
muggalira
muggulira
muggya
mugoba
mugobaganya
mugobera
mugoberera
mugondera
mugula
mugulira
mugulumiza
mugumya
muguza
mugwa
mugya
mujanjaba
mujeemera
mujja
mujjanjaba
mujjanjabira
mujjukira
mujjukiza
mukaka
mukakasa
mukaluubirira
mukambuwalira
mukanda
mukangavvula
mukansa
mukebera
mukema
mukiikirira
mukima
mukka
mukkakkanya
mukkiriza
mukkirizisa
mukola
mukolera
mukomerera
mukoppa
mukosa
mukozesa
mukuba
mukubagiza
mukubanga
mukubira
mukubiriza
mukugira
mukulisa
mukungubagira
mukungula
mukunya
mukuuma
mukuumira
mukuza
mukwana
mukwasa
mukwata
mukwatira
mukwatirira
mukyalira
mukyawa
mukyusa
mulaba
mulabirira
mulabula
mulaga
mulagira
mulamusa
mulanga
mulangira
mulangirira
muleeta
muleetera
muleka
mulekera
mulemesa
mulengezza
muliisa
mulima
mulinda
mulinnya
muliraana
mulisa
muliyirira
muloga
mulokola
mulonda
mulondoola
mulongoosa
muloopa
muloopera
muluma
mulumba
mulumiriza
mulumya
mulungula
mulwanyisa
mulya
mumalira
mumanya
mumatiza
mumenya
mumiza
mumma
mumyuka
munaaza
munafuya
munenya
munnyonnyola
munona
munoonya
munoonyeza
mununula
munyaga
munyiga
munyiiza
munyooma
munyumira
munywa
munywegera
munyweza
mupangisa
muperereza
musaasira
musaba
musabira
musala
musalira
musamba
musanga
musanyusa
musasula
musasulira
musekerera
musemberera
musiba
musibira
musibya
musigulira
musiibula
musiiga
musiima
musika
musikira
musikiriza
musimba
musimbira
musindiikiriza
musindika
musindikira
musinga
musingisa
musinza
musisinkana
musitula
musobola
musobozesa
musojja
musomera
musomesa
musonseka
musonyiwa
musoya
mussa
musunguwaza
musuubiza
musuula
musuuza
musuza
muswaza
muta
mutaasa
mutabukira
mutabula
mutambuza
mutangira
muteebereza
muteega
muteeka
muteekateeka
mutega
mutegeera
mutegeeza
mutegekera
mutema
mutemula
mutendeka
mutendereza
mutiisa
mutiisatiisa
mutikka
mutolosa
mutomera
mutongoza
mutta
muttira
mutuga
mutulugunya
mutuma
mutunda
mutunuulira
mutuuka
mutuukirira
mutuuma
mutuusa
mutuuza
mutwaala
mutwala
mutwalira
mutya
muviira
muvuga
muvuganya
muvuma
muvunaana
muwa
muwaabira
muwabula
muwagira
muwakanya
muwamba
muwandiikira
muwangula
muwasa
muwawaabira
muweebwa
muweerera
muweereza
muweerezanga
muweesa
muwemula
muwerekera
muwereza
muwola
muwolereza
muwondera
muwonya
muwooyawooya
muwozesa
muwujja
muwulira
muwuliriza
muwuliza
muwummuza
muyamba
muyigga
muyiggira
muyigiriza
muyiira
muyimbira
muyimbula
muyimiriza
muyingiza
muyisa
muyita
muyonsa
muyoolayoola
muyozaayoza
muzaala
muzaalira
muziika
muzikiriza
muzimba
muzimbira
muziyiza
muzuukiza
muzuukusa
muzuula
muzza
mwa
mwabulira
mwagala
mwagalira
mwagazisa
mwambaza
mwambula
mwanguyira
mwanika
mwaniriza
mwanjula
mwanjulira
mwanukula
mwawula
mwaza
mwebaza
mwebazanga
mwebwa
mwefuulira
mwegayirira
mwejjeereza
mwekebejja
mwekengera
mwenya
mweraliikiriza
mwesembereza
mwesiga
mwesonyiwa
mwetaagira
mwetegereza
mwetondera
mwetooloola
mwewala
mwewuunyisa
mweyabiza
mweyambisa
mweyimirira
mwogerera
mwogereza
mwogoloza
mwokya
mwolekera
mwongera
mwononekera
mwonoona
mwonoonekera
mwonoonera
myansa
myuka
myumyula
naaba
naabuuka
naatera
naaza
naazibwa
naba
nafuwa
nafuya
nakuwala
nakuwalira
nanagira
ndaba
ndabirira
ndaga
ndeetera
ndeka
ndiisa
ndiyirira
nemesa
nenya
nenyezebwa
nfumbirwa
nfunira
nfunyisa
nfuula
ngaana
ngamba
ngoba
ngobaganya
ngoberera
ngula
njagala
njigiriza
njooga
nkakasa
nkebera
nkola
nkolera
nkozesa
nkuba
nkubira
nkuuma
nkwana
nkwata
nkyalira
nneebaza
nneeraliikiriza
nneewuunyisa
nnemesa
nnoga
nnongoosa
nnoonya
nnuma
nnumba
nnumiriza
nnumya
nnunula
nnyamba
nnyanjula
nnyiiza
nnyika
nnyikira
nnyikiza
nnyogoga
nnyonnyoka
nnyonnyola
nnyonnyolwa
nnyonyoka
nnyonyola
nnyuka
noba
noga
nogera
nogerwa
nona
nonooza
nonya
nonyereza
nonyeza
noonya
noonyereza
noonyerezza
noonyeza
noonyezebwa
nooyereza
nsaba
nsalira
nsanga
nsanyusa
nsasula
nsiba
nsinga
nsinza
nsisinkana
nsomesa
nsonyiwa
nsunguwaza
nsuula
nsuza
nta
ntaasa
nteeka
ntegeeza
ntengula
ntiisatiisa
ntulugunya
ntuma
ntunuulira
ntuusa
ntwala
nunula
nunulibwa
nunulwa
nuuma
nuuna
nuusa
nvuma
nyaga
nyagibwa
nyagulula
nyakula
nyamba
nyeenya
nyenya
nyiga
nyigibwa
nyigiriza
nyigirizibwa
nyiiga
nyiigira
nyiikira
nyiikirira
nyiiza
nyika
nyikira
nyikiza
nyirira
nyiriza
nyiza
nyoga
nyogoga
nyolwa
nyomoola
nyonnyoka
nyonnyola
nyonyogera
nyonyola
nyooka
nyooma
nyoomebwa
nyuka
nyuma
nyumira
nyumirwa
nyumisa
nyumiza
nyumunguza
nyumya
nywa
nywanga
nyweeza
nywegera
nywera
nywerera
nywesa
nyweza
nywezebwa
nzaala
nzigya
nziika
nzijanjaba
nzijira
nzikiriza
nzimbira
nziruukirira
nzisa
nzita
nzitira
nziza
paakinga
pakasa
pakira
pamga
panga
pangisa
pangisibwa
papa
peeka
pepeya
perereza
pima
pimibwa
pimira
pondooka
punta
pya
ranbura
saaba
saabala
saabaza
saabulula
saaga
saagira
saala
saalimbira
saalira
saanga
saanikira
saanuuka
saasaana
saasaanira
saasaanya
saasaanyizibwa
saasana
saasanya
saasira
saasirwa
saawa
saaza
saba
sabanga
sabibwa
sabira
sabiriza
sabirwa
sadaaka
saddaaka
saddaakibwa
sagaasagana
saggula
sajjalaata
sajjuka
sajjula
saka
sakira
sala
salaasala
salanga
salibwa
salimbira
salira
salirwa
salwa
samba
sambira
samira
sanga
sangibwa
sangula
sanikira
sanirira
sanjaga
sannyalala
sannyalaza
sanukula
sanyalala
sanyalaza
sanyuka
sanyukira
sanyusa
sanyusibwa
sasaana
sasaanira
sasaanya
sasamaza
sasana
sasanya
sasika
sasira
sasula
sasulanga
sasulibwa
sasulira
sasulirwa
sasulwa
sasuza
sattira
sattiza
sattulula
sawuka
sawula
saza
seebengerera
seembera
seera
seerera
seetuka
seetula
seeyeya
segeera
seka
sekerera
sekererwa
seketerera
sekula
semba
sembera
semberera
sembereza
sembeza
sembezebwa
sena
senda
sendasenda
senga
sengejja
sengeka
senguka
sengukira
sengula
sengulira
sengulwa
sensera
senserwa
senvula
senya
sereba
sereka
serengeta
sesebbula
sesema
seyeeya
siba
sibibwa
sibira
sibirira
sibirwa
sibisa
sibuka
sibulukuka
sibwa
siga
sigala
sigalira
sigaza
sigukulula
sigula
siguukulula
siiba
siibula
siibulukuka
siiga
siigibwa
siika
siikiriza
siima
siimba
siimbula
siimibwa
siimuula
siinga
siitaana
siiwa
siiwuuka
siiyibwa
sika
sikibwa
sikira
sikiriza
sikirizibwa
sikiza
sima
simattuka
simatuka
simba
simbibwa
simbira
simbuka
simbula
simbuliza
simibwa
simira
simula
simuula
simwa
sinda
sindana
sindiikiriza
sindiikirizibwa
sindika
sindikibwa
sindikira
sindikiriza
sindirira
sindogoma
sinduukirirwa
singa
singana
singibwa
singira
singisa
singisibwa
singiza
sinza
sinzanga
sinzibwa
sinziira
sinzila
sinzira
sinziza
siramuka
siriikirira
siriira
siriiza
sirika
sirikira
sirikirwa
sirisa
sirisibwa
siruwaza
sisimuka
sisinkana
sisinkanira
sitama
situka
situkira
situla
situlanga
situnkana
situula
siwuuka
siyagguka
soba
soberwa
soboka
sobola
soboola
sobozesa
sobya
sogola
sojjolimba
soka
sokera
sokoola
solooza
soma
somanga
somba
somebwa
somera
somerera
somerwa
somesa
somesebwa
somoka
somooza
somoozebwa
somosa
somozebwa
somwa
sonda
sondera
songa
songosa
sonjola
sonsa
sonseka
sonyibwa
sonyiwa
sonyiwagana
sonyiyibwa
sonyiyizibwa
sooba
sooberera
soobola
sooka
sookanga
sookera
sooma
soomooza
soomoozebwa
soomozebwa
soona
soosowaza
soosowazibwa
soowagana
sosola
sosolagana
sosolebwa
sosolwa
sosonkereza
sosowaza
ssa
ssibwa
ssuuka
subula
subwa
sukka
sukkirira
sukkuluma
sula
sumagira
sumba
sumbuwa
sumika
sumulukuka
sumulula
sumululwa
suna
sunda
sunguwala
sunguwalira
sunguwaza
sunsula
sunsulwa
susa
sussa
suubiira
suubira
suubirwa
suubiza
suubula
suubulagana
suubuza
suuka
suula
suulibwa
suulira
suulirira
suulumba
suumusa
suumusibwa
suumuusibwa
suusuuta
suuta
suuza
suza
suzibwa
swaakira
swaaza
swakira
swala
swankula
swanyuula
swaza
swazibwa
taakiriza
taama
taasa
taasibwa
taataaganya
taataaganyizibwa
taataganya
taataganyizibwa
taawulula
taayiza
taba
tabaala
tabagana
tabaganya
tabanguka
tabangula
tabika
tabuka
tabukira
tabula
tabulatabula
tabulwa
tagala
tagambika
taganjula
taggulula
tajjulukuka
takoma
takula
talaaga
talagga
talantuka
talina
tambula
tambulanga
tambulatambula
tambulira
tambuliranga
tambuliza
tambulizibwa
tambuza
tambuzibwa
tamiira
tamiiza
tamwa
tandiika
tandiikira
tandika
tandikibwa
tandikira
tandisa
tanga
tangaala
tangaaza
tangaazibwa
tangaza
tangira
tangirira
tangirwa
tankana
tankanibwa
tanula
tanzibwa
taputa
tasa
tataaganya
tataaganyizibwa
tataganya
tataganyizibwa
tatana
tawaanya
tawaanyizibwa
tawulula
tayirira
tebeereza
tebenkera
tebenkeza
tebenta
tebentesa
tebereza
tebuka
teeba
teebera
teebereza
teeberezebwa
teebwa
teega
teegera
teeka
teekateeka
teekebwa
teekebwateekebwa
teekera
teekerateekera
teekesa
teela
teesa
teeseganya
teeseza
tega
tegana
teganya
tegeela
tegeera
tegeeragana
tegeerebwa
tegeerekeka
tegeerera
tegeerwa
tegeeza
tegeezanga
tegeezebwa
tegeka
tegekebwa
tegekeddwa
tegekera
tegera
tegeza
teka
tekateeka
tekebwa
tekesa
tema
temaatema
temagana
tematema
temula
temulwa
temya
tenda
tendeka
tendekebwa
tendekera
tendereza
tenderezanga
tenderezebwa
tendewalirwa
tengejja
tengula
terebula
tereera
tereeza
tereezebwa
tereka
terekebwa
terekera
terera
tereza
teseganya
tetenkanya
tibwa
tigomya
tiibwa
tiisa
tiisatiisa
tiisibwa
tiisibwatiisibwa
tiitiibya
tijja
tikirwa
tikka
tikkira
tikkirwa
tikkula
timba
timbibwa
timpula
tindigga
tindira
tinta
tirimbula
tisa
tisatiisa
toba
tobeka
toberera
tobya
tojjera
tokosa
tokota
toloka
tomeggana
tomera
tomeragana
tomeregana
tomerwa
tona
tonda
tondebwa
tondeka
tondoka
tonera
tongoza
tongozebwa
tonnya
tontoma
tonya
toola
tottola
tta
ttaanya
ttattana
ttibwa
ttika
ttikirwa
ttira
ttottola
ttukiza
ttulula
ttunka
ttunkana
ttunkira
tubeerera
tubira
tubudaabuda
tubuulira
tubuuza
tubuzaabuza
tuddiza
tudduukirira
tufiirira
tufuga
tufunira
tufuula
tuga
tugamba
tuganyula
tugatta
tuggya
tuguza
tujja
tuka
tukakasa
tukira
tukirira
tukiriza
tukkiriza
tukola
tukolera
tukuba
tukubira
tukubiriza
tukula
tukulembera
tukuuma
tukuumira
tukuza
tukuzibwa
tukwata
tula
tulaba
tulabirira
tulabula
tulaga
tuleetera
tuleka
tulemesa
tuliisa
tulisa
tulokola
tulongoosa
tulugunya
tulugunyizibwa
tulukuka
tuluma
tulumba
tuma
tumbuka
tumbula
tumbulwa
tumibwa
tumya
tunda
tundibwa
tundira
tundula
tunga
tunka
tunnyonnyola
tuntuza
tunula
tunulira
tununula
tunuulira
tunuulirwa
tunuza
tusa
tusaasira
tusaba
tusabira
tusasula
tusinga
tusisinkana
tusobozesa
tusomesa
tusonyiwa
tutaasa
tutabula
tutegeeza
tutendeka
tutiisatiisa
tutta
tutulugunya
tutumuka
tutumula
tutuusa
tutwala
tuuka
tuukana
tuukanya
tuukira
tuukirira
tuukirirwa
tuukiriza
tuukirizibwa
tuula
tuulanga
tuulira
tuuma
tuusa
tuusibwa
tuusiza
tuusizza
tuuyana
tuuza
tuuzanga
tuuzibwa
tuwa
tuweereza
tuwonya
tuwuliriza
tuyamba
tuyigiriza
tuyita
twaala
twaalira
twaaliza
twagala
twala
twalanga
twalibwa
twalira
twalirizibwa
twaliza
twanguyira
twawula
twetaagisa
twewuunyisa
tya
tyaba
tyanga
tyobola
tyoboola
tyoboolebwa
viira
viiraddala
vira
voola
vuba
vubiriza
vubuka
vuga
vugannya
vuganya
vuganyiza
vuganyizibwa
vugira
vugirira
vugisa
vujjirira
vuluga
vuma
vumaganya
vumbeera
vumbula
vumibwa
vumirira
vumula
vunaana
vunaanibwa
vunaanyizibwa
vunama
vunanibwa
vunanyizibwa
vunda
vundira
vunnama
vunnamira
vunuka
vunza
vuunama
vuunika
vuunula
vuvuuma
vuyisa
vva
vvoola
vvunuka
vvunula
vvuunuka
vvuunula
vvuunulwa
yaayaana
yaayaanira
yamba
yambagana
yambanga
yambibwa
yiga
yigga
yigganya
yigganyizibwa
yiggibwa
yiggira
yigira
yigiriza
yigirizibwa
yiibwa
yiika
yiikuula
yiira
yiiya
yiiyiza
yima
yimba
yimbibwa
yimbira
yimbula
yimbulwa
yimira
yimirira
yimiriza
yimirizibwa
yimuka
yimusa
yimusibwa
yinaayina
yindira
yinga
yingira
yingirira
yingiza
yingizibwa
yinza
yiringisa
yisa
yisibwa
yita
yitanga
yitibwa
yitibwanga
yitimuka
yitimusa
yitingana
yitira
yitirira
yitiriza
yitta
yiwa
yiwayiwa
yomba
yombagana
yombayomba
yombera
yongobera
yongobeza
yonja
yonka
yonsa
yoola
yooyoota
yoya
yoyoota
yozaayoza
yozayoza
yubuka
yuguuma
yulika
yunga
yungibwa
yungula
yunja
yuuga
yuuguuma
yuwa
yuza
yuzayuza
zaala
zaalibwa
zaalira
zaalisa
zaama
zaanika
zaawula
zaaza
zala
zalibwa
zambala
zannya
zannyibwa
zannyira
zannyirwa
zannyisa
zanya
zanyibwa
zanyira
zeetegereza
zeewala
zeewola
zeeyambisa
zettanira
ziba
zibawa
zibba
zibikira
zibikula
zibira
zibiriza
zibuka
zibukula
zibula
zibuwaza
zibwa
zibya
ziddaabiriza
zifukirira
zifuna
zifuula
zifuuyira
ziga
zigaba
zigabira
zigema
ziggya
zigonjoola
zigula
ziguza
ziika
ziikibwa
ziikula
ziiyiza
zijjanjaba
zika
zikira
zikirira
zikiriza
zikirizibwa
zikiza
zikola
zikozesa
zikuba
zikuuma
zikuumira
zikwasa
zikwata
zikyusa
zimanya
zimba
zimbibwa
zimbiddwa
zimbira
zimbirwa
zimbulukuka
zimbulukusa
zimbwa
zimenya
zimpa
zimuddiza
zimusasula
zimuwa
zina
zinda
zinga
zingama
zingiza
zingulula
zinira
zinoga
zinoonya
zipangisa
zira
ziraba
zirabirira
zireeta
ziriisa
zirika
zirima
zirondoola
zirongoosa
zirunda
zirwanyisa
zirya
zisa
zisaasaanya
zisala
zisanga
zisasula
zisiba
zisimba
zisindika
zisobozesa
zisoma
zissa
zitambuza
ziteeka
zitegeera
zitegeka
zitereka
zitowa
zitta
zitunda
zituusa
zitwala
ziwa
ziwandiisa
ziwangula
ziweza
ziyamba
ziyira
ziyisa
ziyita
ziyiza
ziyizibwa
zizza
zooza
zula
zuuka
zuukira
zuukiza
zuukizibwa
zuukuka
zuukusa
zuula
zuulibwa
zuulira
zuulwa
zza
zzibwa
ŋŋamba


Writing /tmp/ganda_grammar_bundle/ganda_grammar/rules/verbs.txt


In [14]:
%%writefile /tmp/ganda_grammar_bundle/scripts/build_spellings.py
"""Build src/rules/spellings.tsv: misspellings of vowel length and doubled consonants, and their dictionary spelling.

    pdftotext -layout -f 364 -l 385 "FSI - Luganda Basic Course - Instructor and Student Text.pdf" glossary.txt
    python scripts/build_spellings.py glossary.txt English-Luganda_version_2_0.txt corpus.jsonl [...] > src/rules/spellings.tsv

1. The FSI Luganda Basic Course glossary (1968) gives each word in the standard spelling: long vowels and long
   consonants written double. Entries damaged by OCR are repaired when exactly one corpus word fits.
2. Each glossary word, its plural and its bare form (without the augment) is a correct spelling.
3. A misspelling is a correct spelling with one vowel or consonant lengthened or shortened, seen in the corpus at least
   twice and at least ten times less often, and not a word of its own: not in the English-Luganda dictionary, not a
   listed noun or verb, not English, not a minimal pair from the orthography guide, not in REVIEWED_OUT.
"""
import collections
import json
import re
import sys
from pathlib import Path

from ganda_grammar.rules.agreement import NOUNS  # the package installed: pip install -e .
from ganda_grammar.rules.joining import VERBS

# Both spellings are words (Walusimbi 2015, §1.1.1-1.1.2, §1.3): never correct one into the other
MINIMAL_PAIRS = "lega leega fuka fuuka siba siiba sima siima mala maala sera seera kola koola ekisero ekiseera".split()
# Words the guide spells that a variant could hit: the interrogatives of §1.3(r) (baluwa 'which ones', not bbaluwa 'letter')
GUIDE_WORDS = "kiruwa ziruwa biruwa aluwa baluwa galuwa".split()
# Found by reading the table: a word of its own, or not sure
REVIEWED_OUT = set("""ddaala ggye omuka obba okuweera okugala okusibula buuliri obuuliri okumaala muuka waala amakati
    entebbe ntebbe kusibula kubera emere""".split())
WORD = re.compile(r"[a-zŋ']+")
TOKEN = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*")  # as the rules tokenize, keeping elided words whole


def glossary(path, counts):
    """(word, class or perfect form, English meaning) from the FSI glossary text."""
    head = re.compile(r"^\s*(\(?[^\s(]+\)?)(?:\s*\(([^)]*)\))?\s{2,}(\S.*)$")
    by_len = collections.defaultdict(list)
    for w, n in counts.items():
        if n >= 3:
            by_len[len(w)].append(w)
    out = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        m = head.match(line)
        if not m or "LUGANDA" in line or m[1].lstrip("(").startswith("."):  # page headers, bound stems (.tegeevu)
            continue
        h = m[1].strip("()\"'·/").replace("·", "").replace("''", "")
        h = re.sub(r"^0", "o", h).replace("1", "l").lower().replace(".", "")
        if re.fullmatch(r"[a-zŋ']+", h):  # clean; OCR reads mw as 'row'
            if "row" in h and counts[h.replace("row", "mw")] > counts[h]:
                h = h.replace("row", "mw")
        elif re.search(r"[a-zŋ]{3}", h):  # damaged: junk is a wildcard; keep the repair only when one word fits
            pat = re.compile("".join(re.escape(ch) if WORD.fullmatch(ch) else "[a-zŋ]{0,2}" for ch in h)
                             .replace("row", "(?:mw|row)"))
            fits = sorted(((counts[w], w) for n in range(len(h) - 3, len(h) + 4) for w in by_len[n] if pat.fullmatch(w)),
                          reverse=True)
            if not fits or (len(fits) > 1 and fits[0][0] < 5 * fits[1][0]):
                continue
            h = fits[0][1]
        else:
            continue
        out.append((h, m[2] or "", re.split(r"\s{2,}", m[3])[0].strip()))
    return out


def plural(word, cls):
    """The plural where the FSI class makes it regular (ekyalo KI-BI → ebyalo), else None."""
    for sg, pl, k in (("eki", "ebi", "KI-BI"), ("eky", "eby", "KI-BI"), ("omu", "emi", "MU-MI"), ("omw", "emy", "MU-MI"),
                      ("aka", "obu", "KA-BU")):
        if cls.startswith(k) and word.startswith(sg):
            return pl + word[3:]


def forms(word, cls):
    """The word, its plural where the class makes it regular, and its bare form."""
    out = {word, plural(word, cls)} - {None}
    return out | {w[1:] for w in out if len(w) > 4 and w[0] in "aeo" and w[1] not in "aeiou"}


def variants(w):
    """w with one vowel or consonant made long or short (l, r, w, y are never doubled, §1.2)."""
    out, i = set(), 0
    while i < len(w):
        if w[i].isalpha() and w[i] not in "lrwy":
            if i + 1 < len(w) and w[i + 1] == w[i]:
                out.add(w[:i] + w[i + 1:])
                i += 2
                continue
            out.add(w[:i] + w[i] + w[i:])
        i += 1
    return out


def main(glossary_txt, dictionary_txt, *corpus):
    counts = collections.Counter()
    for path in corpus:
        for line in open(path, encoding="utf-8"):
            counts.update(TOKEN.findall(json.loads(line)["text"]))
    correct = {}
    for word, cls, _ in glossary(glossary_txt, counts):
        for f in forms(word, cls):
            correct[f] = word
    english = {w.strip().lower() for w in open("/usr/share/dict/words") if w.strip().isalpha()}
    dic = set(WORD.findall(Path(dictionary_txt).read_text(encoding="utf-8").lower())) - english
    real = (set(correct) | set(NOUNS) | VERBS | {"oku" + v for v in VERBS} | dic | {w[1:] for w in dic if w[:1] in "aeo"}
            | set(MINIMAL_PAIRS) | {"oku" + w for w in MINIMAL_PAIRS} | set(GUIDE_WORDS))
    rows = sorted({(v, g, counts[v], counts[g]) for g in correct if counts[g] >= 20 for v in variants(g)
                   if counts[v] >= 2 and counts[g] >= 10 * counts[v] and len(v) >= 4
                   and v not in real and v not in english and v not in REVIEWED_OUT}, key=lambda r: (-r[2], r[0]))
    print("# Misspellings of vowel length and doubled consonants (Walusimbi 2015, §1.1.1-1.1.2, §1.2), built by")
    print("# scripts/build_spellings.py from the FSI Luganda Basic Course glossary (1968) and corpus counts.")
    print("# misspelling\tspelling\tcount of the misspelling\tcount of the spelling")
    for v, g, nv, ng in rows:
        print(f"{v}\t{g}\t{nv}\t{ng}")


if __name__ == "__main__":
    main(*sys.argv[1:])


Writing /tmp/ganda_grammar_bundle/scripts/build_spellings.py


In [15]:
%%writefile /tmp/ganda_grammar_bundle/scripts/bundle_notebook.py
"""Write a self-contained copy of a notebook, carrying ganda-grammar in its first cells.

    python scripts/bundle_notebook.py notebooks/build_dictionary.ipynb
    → notebooks/build_dictionary_selfcontained.ipynb (or name the output as a second argument)

The copy starts with one %%writefile cell per file of the package (src/, written as ganda_grammar/, with its data files)
and of scripts/, all under BUNDLE, which goes first on sys.path. Its own cells then find BUNDLE defined, so the notebook
runs with no copy of this repository attached and nothing to install: the rules engine is pure Python.
"""
import json
import subprocess
import sys
from pathlib import Path

REPO = Path(__file__).resolve().parents[1]
BUNDLE = "/tmp/ganda_grammar_bundle"
PARTS = {"src": "ganda_grammar", "scripts": "scripts"}  # repository folder → folder in the bundle


def files():
    for folder, target in PARTS.items():
        for p in sorted((REPO / folder).rglob("*")):
            if p.is_file() and "__pycache__" not in p.parts:
                yield p, f"{BUNDLE}/{target}/{p.relative_to(REPO / folder).as_posix()}"


def cell(kind, text, n):
    c = {"cell_type": kind, "id": f"bundle{n:02d}", "metadata": {}, "source": text.splitlines(keepends=True)}
    if kind == "code":
        c |= {"execution_count": None, "outputs": []}
    return c


def version():
    try:
        rev = subprocess.run(["git", "-C", str(REPO), "describe", "--always", "--dirty"], capture_output=True, text=True,
                             check=True).stdout.strip()
        return f"commit {rev}"
    except (OSError, subprocess.CalledProcessError):
        return "no git revision"


def main(source, out=None):
    source = Path(source)
    out = Path(out) if out else source.with_name(source.stem + "_selfcontained.ipynb")
    nb = json.loads(source.read_text(encoding="utf-8"))
    bundled = list(files())
    folders = sorted({str(Path(target).parent) for _, target in bundled})
    head = [
        cell("markdown", f"""## ganda-grammar, carried by this notebook

This copy of `{source.name}` carries ganda-grammar ({version()}). The cells below write its package (`src/`, as
`ganda_grammar`) and `scripts/` to `{BUNDLE}` and put that first on `sys.path`, so the notebook needs no copy of the
repository as an input. Made by `scripts/bundle_notebook.py`; rebuild it after changing the code.
""", 0),
        cell("code", f"""import sys
from pathlib import Path

BUNDLE = Path("{BUNDLE}")
for folder in {folders!r}:
    Path(folder).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BUNDLE))
""", 1),
    ]
    for p, target in bundled:
        head.append(cell("code", f"%%writefile {target}\n{p.read_text(encoding='utf-8')}", len(head)))
    nb["cells"] = head + nb["cells"]
    out.write_text(json.dumps(nb, indent=1, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"{out}: {len(bundled)} files bundled, {len(nb['cells'])} cells")


if __name__ == "__main__":
    main(*sys.argv[1:])


Writing /tmp/ganda_grammar_bundle/scripts/bundle_notebook.py


In [16]:
%%writefile /tmp/ganda_grammar_bundle/scripts/compile_luganda_text.py
"""Compile the Luganda text under a folder into chunked .txt files, the local input of notebooks/build_dictionary.ipynb.

    python scripts/compile_luganda_text.py ../ ../luganda-dictionary-inputs

Reads every .txt, .jsonl and .html file under the folder (except SKIP) and keeps the sentences that read as Luganda (the
ganda_grammar sentence check). Files are taken cleanest first (largest share of Luganda sentences); a file that is mostly
a copy of one already taken (DUPLICATE of its 5-word sequences seen before, like a re-OCR of the same book) is skipped,
and every sentence is written once. texts/luganda_text_NNN.txt get one paragraph a line, CHUNK_BYTES at most each;
texts/manifest.tsv says what each file gave, or which files were skipped as copies. sources/ gets the text layers of the
PDFs the notebook reads (pdftotext -layout), so it needs neither the PDFs nor poppler.
"""
import hashlib
import json
import re
import shutil
import subprocess
import sys
from html.parser import HTMLParser
from pathlib import Path

from ganda_grammar.rules.base import luganda_tokens, sentences

CHUNK_BYTES = 10_000_000
DUPLICATE = 0.5  # share of a file's 5-word sequences already written that makes it a copy
SKIP = {  # not Luganda (machine-translation and tokenizer experiments), or derived from the files kept here
    "minimind", "cross-lingual-group-tokenizer", "fastest-efficient-tokenizer", "brain.en", "grouptok", "grouptok-paper",
    "ganda-grammar", "luganda-dictionary-inputs", ".git", ".venv", "venv", "node_modules", "site-packages", "__pycache__",
    ".cache", ".ipynb_checkpoints",
}
RESOURCES = "gandalm/resources"
PDF_TEXT = {  # file in sources/: (PDF under RESOURCES, pdftotext page options)
    "fsi_glossary_layout.txt": ("FSI - Luganda Basic Course - Instructor and Student Text.pdf", ["-f", "364", "-l", "385"]),
    "english_luganda_layout.txt": ("English-Luganda_version_2_0.pdf", []),
}
FSI_TEXT = "FSI - Luganda Basic Course - Instructor and Student Text.txt"  # its lesson glossaries


class VisibleText(HTMLParser):  # the text of a page, without scripts and styles
    def __init__(self):
        super().__init__()
        self.parts, self.hidden = [], 0

    def handle_starttag(self, tag, attrs):
        self.hidden += tag in ("script", "style")

    def handle_endtag(self, tag):
        if tag in ("script", "style") and self.hidden:
            self.hidden -= 1

    def handle_data(self, data):
        if not self.hidden:
            self.parts.append(data)


def strings(x):  # every string in a JSON row: text fields, chat turns
    if isinstance(x, str):
        yield x
    elif isinstance(x, dict):
        for v in x.values():
            yield from strings(v)
    elif isinstance(x, list):
        for v in x:
            yield from strings(v)


def paragraphs(path):
    if path.suffix == ".jsonl":
        for line in path.open(encoding="utf-8", errors="replace"):
            try:
                yield from strings(json.loads(line))
            except ValueError:
                yield line
    elif path.suffix == ".html":
        page = VisibleText()
        page.feed(path.read_text(encoding="utf-8", errors="replace"))
        yield from "".join(page.parts).splitlines()
    else:
        yield from path.open(encoding="utf-8", errors="replace")


def files(root):
    for p in sorted(root.rglob("*")):
        if p.suffix in (".txt", ".jsonl", ".html") and p.is_file() and not SKIP.intersection(p.relative_to(root).parts):
            yield p


def luganda_paragraphs(path):
    """The file's paragraphs as lists of their Luganda sentences, and the share of its sentences that are Luganda."""
    out, total = [], 0
    for para in paragraphs(path):
        keep = []
        for s, e in sentences(para):
            total += 1
            if luganda_tokens(para, s, e):
                keep.append(re.sub(r"\s+", " ", para[s:e]).strip())
        if keep:
            out.append(keep)
    return out, sum(map(len, out)) / total if total else 0


def shingles(paras):
    words = " ".join(" ".join(p) for p in paras).lower().split()
    return {hash(tuple(words[i:i + 5])) for i in range(len(words) - 4)}


def main(root, out):
    root, out = Path(root).resolve(), Path(out)
    texts = out / "texts"
    texts.mkdir(parents=True, exist_ok=True)
    found = [(path, *luganda_paragraphs(path)) for path in files(root)]
    seen, written, manifest, n, size = set(), set(), [], 0, CHUNK_BYTES
    chunk = None
    for path, paras, share in sorted(found, key=lambda f: -f[2]):  # cleanest copy first
        if not paras:
            continue
        own = shingles(paras)
        if own and len(own & written) >= DUPLICATE * len(own):
            manifest.append((str(path.relative_to(root)), "copy", "skipped"))
            continue
        written |= own
        kept = words = 0
        for para in paras:
            keep = []
            for sent in para:
                key = hashlib.blake2b(sent.lower().encode(), digest_size=8).digest()
                if key not in seen:
                    seen.add(key)
                    keep.append(sent)
            if not keep:
                continue
            line = " ".join(keep) + "\n"
            if size + len(line.encode()) > CHUNK_BYTES:
                if chunk:
                    chunk.close()
                n += 1
                chunk, size = (texts / f"luganda_text_{n:03d}.txt").open("w", encoding="utf-8"), 0
            chunk.write(line)
            size += len(line.encode())
            kept += len(keep)
            words += len(line.split())
        if kept:
            manifest.append((str(path.relative_to(root)), kept, words))
    if chunk:
        chunk.close()
    with (texts / "manifest.tsv").open("w", encoding="utf-8") as f:
        f.write("file\tsentences\twords\n")
        f.writelines(f"{p}\t{k}\t{w}\n" for p, k, w in manifest)

    sources = out / "sources"
    sources.mkdir(exist_ok=True)
    for name, (pdf, pages) in PDF_TEXT.items():
        subprocess.run(["pdftotext", "-layout", *pages, str(root / RESOURCES / pdf), str(sources / name)], check=True)
    shutil.copy(root / RESOURCES / FSI_TEXT, sources / "fsi_course.txt")

    taken = [m for m in manifest if m[1] != "copy"]
    print(f"{len(taken)} files, {sum(k for _, k, _ in taken):,} sentences, {sum(w for *_, w in taken):,} words → {n} chunks "
          f"in {texts}; {len(manifest) - len(taken)} files skipped as copies")


if __name__ == "__main__":
    main(*sys.argv[1:])


Writing /tmp/ganda_grammar_bundle/scripts/compile_luganda_text.py


# Build a Luganda dictionary from every free source

Merges the word lists and dictionaries below into one Luganda dictionary in the standard spelling and writes it to `dictionary/`:

- `luganda_dictionary.jsonl`: one entry per word: `headword`, `pos`, `noun_class`, `plural`, `english` (meanings), `sources` (which sources list it), `variants` (other spellings the sources used), `count` (occurrences in the texts; none for phrases) and `attested`
- `luganda_dictionary.tsv`: the same, one line per word
- `README.md`: the sources, their licences and what each one contributed

| Source | Gives | Licence |
|---|---|---|
| FSI *Luganda Basic Course* glossary | words, meanings, noun classes, verb perfect forms | public domain (US government) |
| FSI lesson glossaries | words | public domain (US government) |
| English–Luganda word list, version 2.0 | words, meanings | used with permission |
| ganda-grammar noun list and verb stems | words, noun classes | Apache-2.0 |
| [pkyoyetera/luganda_english_dataset](https://huggingface.co/datasets/pkyoyetera/luganda_english_dataset), entries of up to three words | words, meanings | Apache-2.0 (as tagged) |
| [PanLex](https://huggingface.co/datasets/lbourdois/panlex) | words | CC0 |
| [English Wiktionary](https://en.wiktionary.org/wiki/Category:Luganda_lemmas), Luganda lemmas | words, meanings, parts of speech | CC BY-SA 4.0 |
| [Wikidata lexemes](https://www.wikidata.org/wiki/Wikidata:Lexicographical_data) | words, meanings | CC0 |

Texts, used only to count how often each word occurs in Luganda (sentences in other languages are skipped, and each sentence counts once), which decides `attested`:
- the compiled local Luganda text, `texts/luganda_text_*.txt` (made by `scripts/compile_luganda_text.py`);
- [Common Voice Luganda sentences](https://github.com/common-voice/common-voice/tree/main/server/data/lg) (CC0) and the [Makerere English–Luganda parallel corpus](https://zenodo.org/records/4764039) (CC BY 4.0);
- published Hugging Face datasets: [henryhale/luganda-pretrain2](https://huggingface.co/datasets/henryhale/luganda-pretrain2) without `tinystories-ganda`, sagepond/luganda, goldfish fish-food, kukuzaai, MasakhaNEWS and the others in `DATASETS` (the natural-text list of `luganda_unified_corpus.ipynb`; machine-translated ones only with `INCLUDE_SYNTHETIC`).

Scanned books that would need OCR (the Peace Corps manual, Pilkington 1892) are left out.

**Spelling:** every word goes through `ganda_grammar.correct()`, so the orthography rules and the spelling table write it the standard way. The spelling a source used is kept in `variants`.

**Run it on Kaggle** with Internet on and the input `luganda-dictionary-inputs` (the `texts/` and `sources/` folders from `scripts/compile_luganda_text.py`). Use the self-contained version, `build_dictionary_selfcontained.ipynb` from `scripts/bundle_notebook.py`, which carries the ganda-grammar package and scripts in its first cells; this version needs the repository attached as a second input instead. Optional secret: `HF_TOKEN`, for higher Hugging Face rate limits. Saved to `/kaggle/working/dictionary/`.

**Or locally** from the repository, with the package installed (`pip install -e . datasets jupyter`) and `luganda-dictionary-inputs/` next to the repository. `ONLINE = False` uses the local inputs only.


In [17]:
import collections, csv, importlib, io, json, os, random, re, shutil, subprocess, sys, time, urllib.parse, urllib.request
from pathlib import Path

KAGGLE = Path("/kaggle/input").exists()
ONLINE = True              # False: the uploaded inputs only
USE_WIKTIONARY = True      # CC BY-SA: with it, the dictionary as a whole is share-alike
INCLUDE_SYNTHETIC = False  # also count the machine-translated and generated datasets
ROWS_PER_DATASET = None    # None: every row; a number for a quicker run
MIN_COUNT = 2              # a word seen this often in the texts is attested
UA = "ganda-grammar-dictionary/0.1 (Luganda dictionary builder)"


def find(pattern, root):
    hit = next(iter(sorted(root.rglob(pattern))), None)
    assert hit, f"no {pattern} under {root}: attach or place the inputs"
    return hit


# ganda-grammar: carried by this notebook, attached as an input, or the repository this notebook is in
if "BUNDLE" in globals():  # the self-contained version (scripts/bundle_notebook.py) wrote it to BUNDLE
    REPO = BUNDLE
elif KAGGLE:  # this repository attached as an input, installed from a writable copy
    REPO = find("src/rules/base.py", Path("/kaggle/input")).parents[2]
    shutil.copytree(REPO, "/tmp/ganda-grammar", dirs_exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/tmp/ganda-grammar"], check=True)
    importlib.invalidate_caches()
else:  # run from the repository, with the package installed
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if KAGGLE:
    INPUTS, OUT = Path("/kaggle/input"), Path("/kaggle/working/dictionary")
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass  # no secret: anonymous Hugging Face access
else:  # luganda-dictionary-inputs in this folder or one above it
    here = Path.cwd().resolve()
    INPUTS = next((p / "luganda-dictionary-inputs" for p in (here, *here.parents) if (p / "luganda-dictionary-inputs").is_dir()), here)
    OUT = (here if "BUNDLE" in globals() else REPO) / "dictionary"

sys.path.insert(0, str(REPO / "scripts"))
from build_spellings import glossary, plural  # the FSI glossary parser, with its OCR repair
from ganda_grammar import correct
from ganda_grammar.rules.agreement import NOUNS
from ganda_grammar.rules.base import looks_luganda, luganda_tokens, sentences
from ganda_grammar.rules.joining import VERBS

TEXT_CHUNKS = sorted(INPUTS.rglob("luganda_text_*.txt"))
FSI_GLOSSARY = find("fsi_glossary_layout.txt", INPUTS)    # FSI PDF pages 364-385, pdftotext -layout
FSI_COURSE = find("fsi_course.txt", INPUTS)               # the course's text, with its lesson glossaries
WORD_LIST = find("english_luganda_layout.txt", INPUTS)    # the English-Luganda PDF, pdftotext -layout
assert TEXT_CHUNKS, f"no texts/luganda_text_*.txt under {INPUTS}"
OUT.mkdir(parents=True, exist_ok=True)
print(len(TEXT_CHUNKS), "text chunks;", FSI_GLOSSARY.parent)


2 text chunks; /kaggle/input/datasets/henrysml/dict-ganda


In [18]:
# Helpers: downloads, word counts, and the entries being merged
def get(url, **params):
    if params:
        url += "?" + urllib.parse.urlencode(params)
    for attempt in range(4):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": UA}), timeout=120) as r:
                return r.read()
        except Exception:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)


def get_json(url, **params):
    return json.loads(get(url, **params))


TOKEN = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*")
counts = collections.Counter()       # lowercased word → occurrences in the texts
corpus_size = collections.Counter()  # text source → words counted


seen_sentences = set()  # each sentence counts once, whichever text it is found in first


def count_text(source, text):
    """Count the words of the sentences that read as Luganda and haven't been counted yet."""
    words = []
    for s, e in sentences(text):
        key = hash(" ".join(text[s:e].lower().split()))
        if key not in seen_sentences and luganda_tokens(text, s, e):
            seen_sentences.add(key)
            words += [w.lower().replace("’", "'") for w in TOKEN.findall(text[s:e])]
    counts.update(words)
    corpus_size[source] += len(words)


entries = collections.defaultdict(lambda: {"pos": collections.Counter(), "classes": [], "english": [], "sources": set(),
                                           "variants": set(), "plural": None})
added = collections.Counter()  # source → entries it added to
FSI_CLASS = {"MU-BA": "1/2", "MU-MI": "3/4", "N": "9/10", "LI-MA": "5/6", "KI-BI": "7/8", "LU-N": "11/10", "KA-BU": "12/14",
             "BU": "14", "KU-MA": "15/6", "TU": "13", "MA": "6", "LU": "11", "KA": "12", "KI": "7", "LI": "5"}


def clean(word):
    """The word with unified apostrophes and without surrounding punctuation; None if it isn't a plain Luganda word."""
    w = re.sub(r"[’ʼ‘`]", "'", word or "").strip(" \t.,;:!?\"“”()[]{}…*•-")
    w = re.sub(r"\s+", " ", w)
    if w[:1].isupper() and w[1:] == w[1:].lower():
        w = w.lower()  # sentence case in word lists; names that slip in are rare
    if not w or not re.fullmatch(r"[a-zŋ' ]+", w) or len(w) > 60:
        return None
    # every word is Luganda-shaped; an elided prefix (ky' of ky'okulya) is left out of the check
    return w if all(looks_luganda(re.sub(r"^\w{1,3}'", "", part)) for part in w.split()) else None


def add(word, source, pos=None, cls=None, english=(), plural_=None):
    raw = clean(word)
    if not raw:
        return
    head = correct(raw).text
    e = entries[head]
    if pos:
        e["pos"][pos] += 1
    if cls and cls not in e["classes"]:
        e["classes"].append(cls)
    known = {x.lower() for x in e["english"]}
    for en in english:
        en = re.sub(r"\s+", " ", en or "").strip(" .,;")
        if en and en.lower() not in known:
            e["english"].append(en)
            known.add(en.lower())
    if raw != head:
        e["variants"].add(raw)
    if plural_ and not e["plural"]:
        e["plural"] = correct(plural_).text
    e["sources"].add(source)
    added[source] += 1


## 1. Count words in the texts

Counts come first: the FSI glossary repair uses them, and they decide which words are `attested`.


In [19]:
# The compiled local text: Luganda sentences only, one paragraph a line
for p in TEXT_CHUNKS:
    for line in p.open(encoding="utf-8"):
        count_text("local", line)
print(f"local texts: {corpus_size['local']:,} words")


local texts: 1,856,114 words


In [20]:
def j(*xs):  # join the non-empty fields of a row
    return "\n".join(x for x in xs if x)


# Published Luganda datasets: the list of luganda_unified_corpus.ipynb. syn marks machine-translated or generated text.
DATASETS = [
    dict(repo="henryhale/luganda-pretrain2", config="default", cols=["text", "source"], text=lambda r: r["text"],
         syn=lambda r: r["source"] == "tinystories-ganda"),
    dict(repo="sagepond/luganda", config="default", cols=["luganda"], text=lambda r: r["luganda"]),
    dict(repo="goldfish-models/fish-food", config="lug_latn", cols=["text"], text=lambda r: r["text"]),
    dict(repo="kukuzaai/lug-cleaned-200M-tokens", config="default", cols=["headline", "full_body"],
         text=lambda r: j(r["headline"], r["full_body"])),
    dict(repo="sulaimank/luganda_text_wiki_bukedde_radio_flores", config="default", cols=["text"], text=lambda r: r["text"]),
    dict(repo="masakhane/masakhanews", config="lug", cols=["headline", "text"], text=lambda r: j(r["headline"], r["text"])),
    dict(repo="masakhane/masakhaner2", config="lug", cols=["tokens"], text=lambda r: " ".join(r["tokens"] or [])),
    dict(repo="michsethowusu/english-ganda_sentence-pairs_mt560", config="default", cols=["lug"], text=lambda r: r["lug"]),
    dict(repo="aceuganda/lacuna-luganda-health-corpus", config="default", cols=["approved_translation"],
         text=lambda r: r["approved_translation"]),
    dict(repo="pkyoyetera/luganda_english_dataset", config="default", cols=["Luganda"], text=lambda r: r["Luganda"]),
    dict(repo="KMayanja/merged_txt_eng_lug", config="default", cols=["Luganda"], text=lambda r: r["Luganda"]),
    dict(repo="masakhane/afriscience_mt", config="corpus", cols=["lang_pair", "target"],
         keep=lambda r: r["lang_pair"] == "eng-lug", text=lambda r: r["target"]),
    dict(repo="BUAIR/Uganda-Multilingual-QA", config="luganda", cols=["question", "answer"],
         text=lambda r: j(r["question"], r["answer"])),
    dict(repo="akoksal/muri-it-language-split", config="lug", cols=["output"], text=lambda r: r["output"]),
    dict(repo="ayymen/Pontoon-Translations", config="en-lg", cols=["target_string"],
         text=lambda r: (r["target_string"] or "").strip('"')),
    dict(repo="allandclive/Luganda_Sci-Math-Bio_Translations", config="default", cols=["Luganda"], text=lambda r: r["Luganda"]),
    # speech transcripts (only the text column is read, never the audio)
    dict(repo="keystats/luganda_asr_dataset", config="default", cols=["transcription"], text=lambda r: r["transcription"]),
    dict(repo="google/WaxalNLP", config="lug_asr", cols=["transcription"], text=lambda r: r["transcription"]),
    # machine-translated or generated
    dict(repo="saillab/alpaca-luganda-cleaned", config="default", cols=["instruction", "input", "output"], syn=True,
         text=lambda r: j(r["instruction"], r["input"], r["output"])),
    dict(repo="henryhale/luganda-sft-translated", config="default", cols=["instruction_lug", "output_lug"], syn=True,
         text=lambda r: j(r["instruction_lug"], r["output_lug"])),
    dict(repo="CraneAILabs/luganda-bilingual-literacy-exercises", config="all", syn=True,
         cols=["luganda_question", "luganda_answer", "luganda_explanation"],
         text=lambda r: j(r["luganda_question"], r["luganda_answer"], r["luganda_explanation"])),
]

if ONLINE:
    # Common Voice Luganda sentences (CC0)
    for f in get_json("https://api.github.com/repos/common-voice/common-voice/contents/server/data/lg"):
        if f["name"].endswith(".txt"):
            count_text("common-voice", get(f["download_url"]).decode("utf-8"))
    # Makerere English-Luganda parallel corpus (CC BY 4.0): its Luganda side; a few bytes aren't UTF-8
    for row in csv.DictReader(io.StringIO(get("https://zenodo.org/records/4764039/files/Luganda.csv?download=1").decode("utf-8-sig", errors="replace"))):
        count_text("makerere-parallel", row.get("Luganda") or "")
    from datasets import load_dataset
    for d in DATASETS:
        if d.get("syn") is True and not INCLUDE_SYNTHETIC:
            continue
        t, n = time.time(), 0
        try:
            for row in load_dataset(d["repo"], d["config"], split="train", streaming=True).select_columns(d["cols"]):
                if callable(d.get("syn")) and d["syn"](row) and not INCLUDE_SYNTHETIC:
                    continue
                if "keep" in d and not d["keep"](row):
                    continue
                count_text(d["repo"], d["text"](row) or "")
                n += 1
                if n == ROWS_PER_DATASET:
                    break
            print(f"{d['repo']:52} {n:>10,} rows {corpus_size[d['repo']]:>14,} new words {time.time() - t:6.0f}s")
        except Exception as e:  # gone, renamed, gated: the others still count
            print(f"{d['repo']:52} skipped: {type(e).__name__}: {str(e)[:120]}")
for source, n in corpus_size.items():
    print(f"{source:52} {n:>14,} words")
print(f"{len(counts):,} distinct words")


README.md:   0%|          | 0.00/2.98k [00:00<?, ?B/s]

henryhale/luganda-pretrain2                           1,411,763 rows     27,181,836 new words   1119s


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

sagepond/luganda                                        986,379 rows      9,414,173 new words    134s


README.md:   0%|          | 0.00/30.7k [00:00<?, ?B/s]

goldfish-models/fish-food                             2,036,594 rows      6,051,516 new words     83s


... (31 outputs left out)

README.md:   0%|          | 0.00/36.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/67 [00:00<?, ?it/s]

google/WaxalNLP                                           5,455 rows        157,282 new words     51s
local                                                     1,856,114 words
common-voice                                              1,340,481 words
makerere-parallel                                            24,044 words
henryhale/luganda-pretrain2                              27,181,836 words
sagepond/luganda                                          9,414,173 words
goldfish-models/fish-food                                 6,051,516 words
kukuzaai/lug-cleaned-200M-tokens                         15,627,676 words
sulaimank/luganda_text_wiki_bukedde_radio_flores          4,242,875 words
masakhane/masakhanews                                       122,946 words
michsethowusu/english-ganda_sentence-pairs_mt560          2,759,983 words
aceuganda/lacuna-luganda-health-corpus                      644,535 words
pkyoyetera/luganda_english_dataset                           37,286 words
KMayanja/m

## 2. The word lists


In [21]:
# FSI Luganda Basic Course glossary: the text layer of its PDF pages (no OCR), parsed and repaired by build_spellings
for word, extra, gloss in glossary(FSI_GLOSSARY, counts):
    if extra.startswith("."):  # a verb with its perfect form: okwagala (.agadde)
        add(word, "fsi-glossary", pos="verb", english=[gloss])
    else:
        cls = FSI_CLASS.get(extra.upper().replace(" ", ""))
        add(word, "fsi-glossary", pos="noun" if cls else None, cls=cls, english=[gloss], plural_=plural(word, extra))

# The lesson glossaries: only the headwords (dot notation, o.mu.ntu); their meanings are out of order in the scan
for block in FSI_COURSE.read_text(encoding="utf-8", errors="replace").split("Glossary:")[1:]:
    for line in re.split(r"LUGANDA BASIC COURSE|LESSON\s+\d+", block)[0].splitlines():
        h = line.strip()
        if re.fullmatch(r"[a-zŋ]+(?:\.[a-zŋ]+)+", h):
            w = h.replace(".", "")
            if "row" in w and counts[w.replace("row", "mw")] > counts[w]:  # OCR reads mw as 'row'
                w = w.replace("row", "mw")
            add(w, "fsi-lessons")
print(added)


Counter({'fsi-glossary': 342, 'fsi-lessons': 99})


In [22]:
# English-Luganda word list v2.0: eight columns a page of "english luganda, luganda" entries, the Luganda often wrapped.
# The text layer of its PDF (not a scan) with the layout kept, cut into columns: a column starts where most lines start a word.
def columns(page):
    lines = [l for l in page.splitlines() if l.strip()]
    hits = collections.Counter(m.start() for l in lines for m in re.finditer(r"\S+", l)
                               if m.start() <= 1 or l[m.start() - 2:m.start()] == "  ")
    starts = []
    for p in sorted(p for p, n in hits.items() if n >= max(4, len(lines) // 5)):
        if not starts or p - starts[-1] > 3:
            starts.append(p)
    cols = [[] for _ in starts]
    for line in lines:
        prev = 0
        for c, p in enumerate(starts[1:] + [len(line)]):
            p = min(p, len(line))
            while prev < p < len(line) and line[p - 1] != " " and line[p] != " ":
                p -= 1  # a word running over the column edge belongs to the next column
            cols[c].append(line[prev:p].strip())
            prev = p
    return cols


layout = WORD_LIST.read_text(encoding="utf-8")
column_lines = [line for page in layout.split("\f") for col in columns(page) for line in col if line]
dictionary_words = Path("/usr/share/dict/words")
ENGLISH = {w.strip().lower() for w in dictionary_words.open()} if dictionary_words.exists() else set()


FUNCTION_WORDS = {"a", "an", "the", "to", "of", "in", "on", "up", "out", "off", "for", "with", "by", "be", "as", "at"}


def english(tok, in_headword):
    """An English word: in the English word list, and not Luganda-shaped or rarely used in Luganda (apologize, but ani)."""
    t = tok.lower().strip(",.;()")
    if not ENGLISH:
        return not looks_luganda(t)
    return t in ENGLISH and (not looks_luganda(t) or counts[t] < 5 or (in_headword and t in FUNCTION_WORDS))


pairs = []  # [english, luganda]; a line can hold two entries when the PDF's columns run together
for line in column_lines:
    toks = line.split()
    for i, tok in enumerate(toks):
        in_headword = bool(pairs) and i > 0 and not pairs[-1][1]
        if english(tok, in_headword) and in_headword:
            pairs[-1][0] += " " + tok  # the English headword goes on: bus stop, according to
        elif english(tok, in_headword):
            pairs.append([tok, ""])  # a new entry, at the start of a line or after the Luganda of the last one
        elif pairs:
            pairs[-1][1] += " " + tok
for en, lg in pairs:
    for alt in re.split(r"[,;/]| oba ", lg):
        add(alt, "english-luganda-list", pos="verb" if alt.strip().startswith(("oku", "okw")) else None, english=[en])
print(len(pairs), "English headwords;", added["english-luganda-list"], "Luganda words")

# ganda-grammar's own lists: the agreement nouns and the verb stems
for noun, cls in NOUNS.items():
    add(noun, "ganda-grammar-nouns", pos="noun", cls=cls)
# The stems were counted from text, so some carry an object prefix (okumuleka 'to leave him' = mu + leka): skip those
OBJECT = re.compile(r"(?:n|m|mu|ku|tu|ba|gu|gi|li|ga|ki|bi|lu|zi|ka|bu|wa)(?=[^aeiou])")
for stem in VERBS:
    m = OBJECT.match(stem)
    if not (m and len(stem) - m.end() >= 3 and stem[m.end():] in VERBS):
        add("oku" + stem, "ganda-grammar-verbs", pos="verb")
print(added)


2831 English headwords; 3309 Luganda words
Counter({'english-luganda-list': 3309, 'ganda-grammar-verbs': 2181, 'fsi-glossary': 342, 'ganda-grammar-nouns': 230, 'fsi-lessons': 99})


In [23]:
if ONLINE:
    # pkyoyetera/luganda_english_dataset: the rows of up to three words on each side are dictionary entries
    from datasets import load_dataset
    for row in load_dataset("pkyoyetera/luganda_english_dataset", split="train"):
        en, lg = (row["English"] or "").strip(), (row["Luganda"] or "").strip()
        if 0 < len(en.split()) <= 3 and 0 < len(lg.split()) <= 3:
            for alt in re.split(r"[,;/]", lg):
                add(alt, "pkyoyetera-pairs", pos="verb" if alt.strip().startswith(("oku", "okw")) else None, english=[en])
    print(added["pkyoyetera-pairs"])


data/train-00000-of-00001-d6ac85019c81ba(…): reconstructing file:   0%|          |  0.00B / 7.02MB            

data/train-00000-of-00001-d6ac85019c81ba(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78238 [00:00<?, ? examples/s]

16509


In [24]:
# PanLex (CC0): the Luganda rows, through the Hugging Face datasets server; else the whole CSV is read once (1.3 GB)
PANLEX = dict(dataset="lbourdois/panlex", config="panlex", split="train")
SERVER = "https://datasets-server.huggingface.co"


def panlex_columns(cols):
    lang = next(c for c in cols if "639" in c)  # the ISO 639-3 code
    word = [c for c in cols if c != lang and not any(k in c.lower() for k in ("639", "name", "code"))]
    assert word, f"no word column among {cols}: set it by hand"
    print("PanLex columns:", cols, "→ language:", lang, "word:", word[0])
    return lang, word[0]


def panlex_luganda():
    try:
        lang, word = panlex_columns([f["name"] for f in get_json(f"{SERVER}/first-rows", **PANLEX)["features"]])
        offset, total = 0, None
        while total is None or offset < total:
            page = get_json(f"{SERVER}/filter", **PANLEX, where=f"\"{lang}\"='lug'", offset=offset, length=100)
            total = page["num_rows_total"]
            rows = [r["row"][word] for r in page["rows"]]
            if not rows:
                break
            yield from rows
            offset += len(rows)
    except Exception as e:
        print("datasets server unavailable:", e, "→ reading panlex.csv")
        url = "https://huggingface.co/datasets/lbourdois/panlex/resolve/main/panlex.csv"
        with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": UA}), timeout=600) as r:
            rows = csv.reader(io.TextIOWrapper(r, encoding="utf-8"), delimiter=";")
            header = next(rows)
            lang, word = panlex_columns(header)
            li, wi = header.index(lang), header.index(word)
            yield from (row[wi] for row in rows if len(row) > max(li, wi) and row[li] == "lug")


if ONLINE:
    for w in panlex_luganda():
        add(w, "panlex", pos="verb" if w.startswith(("oku", "okw")) else None)
    print(added["panlex"])


PanLex columns: ['vocab', '639-3', '639-3_english_name', 'var_code', 'english_name_var'] → language: 639-3 word: vocab
datasets server unavailable: HTTP Error 500: Internal Server Error → reading panlex.csv
PanLex columns: ['vocab', '639-3', '639-3_english_name', 'var_code', 'english_name_var'] → language: 639-3 word: vocab
7096


In [25]:
WIKTIONARY = "https://en.wiktionary.org/w/api.php"


def plain(wikitext):
    """A definition line without wiki markup: [[a|b]] → b, {{l|en|x}} → x, other templates dropped."""
    s = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]*)\]\]", r"\1", wikitext)
    s = re.sub(r"\{\{(?:l|m|ll|gloss)\|(?:[^|}]*\|)?([^|}]*)[^}]*\}\}", r"\1", s)
    s = re.sub(r"\{\{[^{}]*\}\}", "", s)
    return re.sub(r"'{2,3}|<[^>]+>", "", s).strip(" :;,")  # bold/italic quotes and HTML tags


if ONLINE and USE_WIKTIONARY:
    titles, more = [], {}
    while True:
        page = get_json(WIKTIONARY, action="query", list="categorymembers", cmtitle="Category:Luganda lemmas",
                        cmtype="page", cmlimit=500, format="json", **more)
        titles += [m["title"] for m in page["query"]["categorymembers"]]
        if "continue" not in page:
            break
        more = page["continue"]
    for i in range(0, len(titles), 50):
        page = get_json(WIKTIONARY, action="query", prop="revisions", rvprop="content", rvslots="main", format="json",
                        formatversion=2, titles="|".join(titles[i:i + 50]))
        for p in page["query"]["pages"]:
            text = p["revisions"][0]["slots"]["main"]["content"]
            m = re.search(r"^==\s*Luganda\s*==\s*$(.*?)(?=^==[^=]|\Z)", text, re.M | re.S)
            if not m:
                continue
            section = m[1]
            pos = re.search(r"^===+\s*(Noun|Verb|Adjective|Adverb|Pronoun|Conjunction|Preposition|Interjection|Numeral|Determiner)\s*===+",
                            section, re.M)
            cls = re.search(r"\{\{lg-noun\|(\d+)|class\s*=\s*(\d+)", section)
            senses = [plain(g) for g in re.findall(r"^#(?![:*])\s*(.+)$", section, re.M)]
            add(p["title"], "wiktionary", pos=pos[1].lower() if pos else None, cls=cls and (cls[1] or cls[2]),
                english=[s for s in senses if s])
    print(len(titles), "lemmas →", added["wiktionary"])


165 lemmas → 162


In [26]:
if ONLINE:
    query = """SELECT ?lemma ?category ?gloss WHERE {
      ?l dct:language wd:Q33368 ; wikibase:lemma ?lemma ; wikibase:lexicalCategory ?c .
      ?c rdfs:label ?category . FILTER(LANG(?category) = "en")
      OPTIONAL { ?l ontolex:sense ?s . ?s skos:definition ?gloss . FILTER(LANG(?gloss) = "en") }
    }"""
    for b in get_json("https://query.wikidata.org/sparql", query=query, format="json")["results"]["bindings"]:
        add(b["lemma"]["value"], "wikidata", pos=b["category"]["value"].lower(),
            english=[b["gloss"]["value"]] if "gloss" in b else [])
    print(added["wikidata"])


9


## 3. Merge and write


In [27]:
SOURCES = {
    "fsi-glossary": ("FSI Luganda Basic Course (1968), glossary", "public domain (US government work)"),
    "fsi-lessons": ("FSI Luganda Basic Course (1968), lesson glossaries", "public domain (US government work)"),
    "english-luganda-list": ("English-Luganda word list, version 2.0", "used with permission"),
    "ganda-grammar-nouns": ("ganda-grammar: nouns of the agreement rules", "Apache-2.0"),
    "ganda-grammar-verbs": ("ganda-grammar: verb stems (corpus counts and the English-Luganda list)", "Apache-2.0"),
    "pkyoyetera-pairs": ("pkyoyetera/luganda_english_dataset, entries of up to three words", "Apache-2.0 (as tagged)"),
    "panlex": ("PanLex, via lbourdois/panlex", "CC0-1.0"),
    "wiktionary": ("English Wiktionary, Category:Luganda lemmas", "CC BY-SA 4.0"),
    "wikidata": ("Wikidata lexemes, language Q33368", "CC0-1.0"),
}
TEXTS = {  # the rest are Hugging Face datasets, named by their repository
    "local": "the compiled local Luganda text (scripts/compile_luganda_text.py)",
    "common-voice": "Common Voice Luganda sentences (CC0-1.0)",
    "makerere-parallel": "Makerere English-Luganda parallel corpus (CC BY 4.0)",
    "henryhale/luganda-pretrain2": "henryhale/luganda-pretrain2 without tinystories-ganda",
}

rows = []
for head, e in sorted(entries.items()):
    count = None if " " in head else counts[head]
    pos = e["pos"].most_common(1)[0][0] if e["pos"] else ("verb" if head.startswith(("oku", "okw")) else None)
    rows.append({"headword": head, "pos": pos, "noun_class": e["classes"][0] if e["classes"] else None,
                 "plural": e["plural"], "english": e["english"], "sources": sorted(e["sources"]),
                 "variants": sorted(e["variants"]), "count": count,
                 "attested": (count or 0) >= MIN_COUNT or len(e["sources"]) >= 2})

with (OUT / "luganda_dictionary.jsonl").open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
with (OUT / "luganda_dictionary.tsv").open("w", encoding="utf-8") as f:
    f.write("headword\tpos\tnoun_class\tplural\tenglish\tsources\tvariants\tcount\tattested\n")
    for r in rows:
        f.write("\t".join(["; ".join(v) if isinstance(v, list) else "" if v is None else str(v) for v in r.values()]) + "\n")

attested = sum(r["attested"] for r in rows)
print(f"{len(rows):,} entries, {attested:,} attested, {sum(bool(r['english']) for r in rows):,} with an English meaning")


15,655 entries, 11,496 attested, 10,590 with an English meaning


In [28]:
only = collections.Counter(r["sources"][0] for r in rows if len(r["sources"]) == 1)
table = "\n".join(f"| {SOURCES[s][0]} | {SOURCES[s][1]} | {added[s]:,} | {only[s]:,} |" for s in SOURCES if added[s])
texts = "\n".join(f"| {TEXTS.get(s, s)} | {n:,} |" for s, n in corpus_size.items())
share_alike = ("Because it includes Wiktionary entries, the dictionary as a whole is under **CC BY-SA 4.0**; "
               "build it with `USE_WIKTIONARY = False` for a version without share-alike."
               if added["wiktionary"] else "No share-alike source is included.")
(OUT / "README.md").write_text(f"""# Luganda dictionary

{len(rows):,} Luganda words and phrases in the standard spelling, {attested:,} of them attested (seen at least {MIN_COUNT} times in the
texts below, or listed by two sources). Built by `notebooks/build_dictionary.ipynb` in the ganda-grammar repository.

Each entry: `headword`, `pos`, `noun_class`, `plural`, `english` (meanings), `sources`, `variants` (spellings the sources
used before the standard spelling was applied), `count` (occurrences in the texts; none for phrases) and `attested`.
Spellings were standardised with the ganda-grammar rules (Walusimbi, *Empandiika y'Oluganda Entongole*, 2015).

## Sources

| Source | Licence | Entries from it | Found only there |
|---|---|---|---|
{table}

Texts used only for counts:

| Text | Words counted (each sentence once) |
|---|---|
{texts}

## Licence

Each entry lists its sources, so it can be traced to its licence. {share_alike}
""", encoding="utf-8")
print((OUT / "README.md").read_text()[:1500])


# Luganda dictionary

15,655 Luganda words and phrases in the standard spelling, 11,496 of them attested (seen at least 2 times in the
texts below, or listed by two sources). Built by `notebooks/build_dictionary.ipynb` in the ganda-grammar repository.

Each entry: `headword`, `pos`, `noun_class`, `plural`, `english` (meanings), `sources`, `variants` (spellings the sources
used before the standard spelling was applied), `count` (occurrences in the texts; none for phrases) and `attested`.
Spellings were standardised with the ganda-grammar rules (Walusimbi, *Empandiika y'Oluganda Entongole*, 2015).

## Sources

| Source | Licence | Entries from it | Found only there |
|---|---|---|---|
| FSI Luganda Basic Course (1968), glossary | public domain (US government work) | 342 | 47 |
| FSI Luganda Basic Course (1968), lesson glossaries | public domain (US government work) | 99 | 10 |
| English-Luganda word list, version 2.0 | used with permission | 3,309 | 391 |
| ganda-grammar: nouns of the ag

In [29]:
# A look at the result: random attested entries, and the words most sources agree on
random.seed(0)
for r in random.sample([r for r in rows if r["attested"]], 15):
    print(f"{r['headword']:22} {r['pos'] or '':10} {r['noun_class'] or '':6} {', '.join(r['english'][:2])[:50]:50} {r['sources']}")
print()
for r in sorted(rows, key=lambda r: (-len(r["sources"]), -(r["count"] or 0)))[:15]:
    print(f"{r['headword']:22} {len(r['sources'])} sources, count {r['count']}")


okucupula              verb                                                                 ['ganda-grammar-verbs']
okukakana              verb                                                                 ['ganda-grammar-verbs']
bulaaya                                  Europe                                             ['english-luganda-list', 'panlex', 'pkyoyetera-pairs']
kuzina                                                                                      ['panlex']
okusoboka              verb              manage                                             ['ganda-grammar-verbs', 'pkyoyetera-pairs']
okunyuka               verb                                                                 ['ganda-grammar-verbs']
okuggweera             verb                                                                 ['ganda-grammar-verbs']
mukkiriza                                                                                   ['panlex']
okumwogereza           verb                    